# 06 · Comparación descriptiva y auditoría de resultados
**Piloto de desarrollo · exclusivamente en Google Colab · CPU.**

Reúne Gurobi, ACO, Tabú, AM supervisado (exploratorio), POMO y HBG-FACS
(adaptación propia), además de EDD, OR-Tools Routing, POMO sin entrenar, política
aleatoria con máscaras, HBG sin feromona y HBG sin entrenar.
Verifica banco, corridas y compatibilidad antes de comparar; no entrena, construye
rutas, ejecuta solvers ni vuelve a simular escenarios. No requiere GPU ni licencias.

El banco aceptado contiene **cuatro geometrías independientes**, una por tamaño.
Semillas, escenarios, políticas y modelos no multiplican N. Este notebook produce
comparaciones de desarrollo y diagnósticos descriptivos: no p-valores, intervalos
poblacionales, bootstrap, rankings globales ni una conclusión de superioridad general.

Los métodos optimizan el objetivo determinista conservador de 01–05; FR, CVaR y
recurso causal se miden con sus exportaciones del evaluador común. No se interpreta
viaje como el costo completo de dos etapas `c+Q`, todavía pendiente de definición.

**Flujo:** verificar banco → pruebas de contrato Colab → inventario de corridas →
selección inequívoca por método → compatibilidad y cobertura → agregación por
geometría → comparación descriptiva → reporte sellado y copia en Drive.

No se elige automáticamente la corrida más reciente ni la de mejores métricas.
Cuando existen varios candidatos, se exige fijar `RUN_SELECTION` por ID antes
de construir el reporte. Los fallos y faltantes permanecen visibles.


## 1. Configuración y entorno
El banco se fija mediante `configs/accepted_bank_00.json`, igual que en 01–05.
Los parámetros de comparación se imprimen a continuación. Fijarlos antes de observar
contrastes; conservar la selección de corridas en el registro del análisis.

`RUN_SELECTION` puede quedar vacío cuando hay un único candidato por método.
Si hay ambigüedad, revisar el inventario y completar el mapping con IDs conocidos;
por ejemplo, `{'aco': 'ID_COPIADO_DEL_INVENTARIO'}`. No inventar IDs ni elegir por
FR/costo. Ejecutar de nuevo la celda de configuración editada y después la celda
de selección; no es necesario repetir las pruebas si no cambió código o configuración.


In [1]:
import sys, os, json, shutil, subprocess, importlib.metadata
from pathlib import Path
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError('Por instrucción del autor: ejecutar exclusivamente en Google Colab.') from exc
BANK_REFERENCE = {'bank_id': 'pilot00_280553a33c2ab594', 'protocol_version': 'pilot-00-v4', 'manifest_sha256': 'd8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02', 'acceptance_status': 'ready_for_method_development', 'role': 'development_only', 'validation_run_id': '20260917T021417Z_665a3a34', 'evidence': 'docs/NOTEBOOK_00_VALIDATION.md'}
BANK_ID = BANK_REFERENCE['bank_id']
SUPPORTED_BANK_PROTOCOLS = (BANK_REFERENCE['protocol_version'],)
RUN_SELECTION = {}
WORK = Path('/content/TESIS_COMPARISON_06')
RUNTIME = WORK / 'runtime'
(RUNTIME / 'svrplab').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'tests').mkdir(exist_ok=True)
(RUNTIME / 'svrplab/__init__.py').write_text('"""Métodos y evaluador sellado del banco de tesis."""\n')
os.chdir(RUNTIME)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
needed = ['numpy', 'pandas', 'matplotlib', 'pytest']
missing = [name for name in needed if version(name) is None]
if missing:
    constraints = RUNTIME / 'constraints.txt'
    protected = ['numpy','pandas','matplotlib','scipy','protobuf','torch','pillow']
    constraints.write_text(''.join(f'{name}=={version(name)}\n' for name in protected if version(name)))
    requirements = missing
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints), *requirements], check=True)
print('Python:', sys.version)
print('Dependencias:', {name: version(name) for name in needed})
print('CPU: análisis de exportaciones existentes, sin entrenamiento ni simulación.')


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Dependencias: {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.0', 'pytest': '8.4.2'}
CPU: análisis de exportaciones existentes, sin entrenamiento ni simulación.


In [2]:
COMPARISON_CONFIG = {'schema_version': 'comparison-06-v2', 'role': 'development_only', 'analysis_mode': 'descriptive_only', 'expected_methods': ['gurobi', 'aco', 'tabu', 'am_supervised', 'pomo', 'hbg_facs', 'edd', 'ortools_routing', 'pomo_untrained', 'random_masked', 'hbg_no_pheromone', 'hbg_untrained'], 'reference_method': 'gurobi', 'metrics': ['scenario_strict_feasibility', 'mean_travel_minutes', 'cvar_travel_minutes', 'mean_lateness_minutes', 'mean_depot_overtime_minutes', 'search_total_seconds'], 'accepted_bank': {'bank_id': 'pilot00_280553a33c2ab594', 'protocol_version': 'pilot-00-v4', 'manifest_sha256': 'd8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02', 'acceptance_status': 'ready_for_method_development', 'role': 'development_only', 'validation_run_id': '20260917T021417Z_665a3a34', 'evidence': 'docs/NOTEBOOK_00_VALIDATION.md'}, 'method_contract': {'schema_version': 'methods-v2', 'role': 'development_only', 'objective': 'conservative_upper_travel_minutes_v1', 'time_scale': 1000, 'seeds': [11, 29, 47], 'search_seconds': 120.0, 'scenario_split': 'evaluation', 'export_all_scenario_events': True, 'methods': {'aco': {'max_iterations': None, 'ants': 20, 'alpha': 1.0, 'beta': 2.0, 'evaporation': 0.1, 'q0': 0.2, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'tabu': {'max_iterations': None, 'neighborhood_samples': 80, 'tenure': 15, 'initial_restarts': 4, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'gurobi': {'threads': 1, 'mip_gap': 0.0}, 'am_supervised': {'n_samples': 16, 'permute_samples': False, 'completion_search_nodes': 2048}, 'pomo': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'hbg_facs': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'edd': {}, 'ortools_routing': {'capacity_scale': 1000}, 'pomo_untrained': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'random_masked': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'hbg_no_pheromone': {'ants': 16, 'iterations': 100, 'alpha': 0.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'hbg_untrained': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}}}, 'controlled_contrasts': [{'method': 'pomo', 'reference': 'pomo_untrained'}, {'method': 'pomo', 'reference': 'random_masked'}, {'method': 'hbg_facs', 'reference': 'hbg_no_pheromone'}, {'method': 'hbg_facs', 'reference': 'hbg_untrained'}, {'method': 'hbg_facs', 'reference': 'ortools_routing'}, {'method': 'hbg_facs', 'reference': 'edd'}], 'method_roles': {'gurobi': 'comparator', 'aco': 'comparator', 'tabu': 'comparator', 'am_supervised': 'exploratory_secondary', 'pomo': 'comparator', 'hbg_facs': 'proposed_declared_adaptation', 'edd': 'comparator', 'ortools_routing': 'comparator', 'pomo_untrained': 'ablation', 'random_masked': 'ablation', 'hbg_no_pheromone': 'ablation', 'hbg_untrained': 'ablation'}, 'future_confirmatory_design': {'enabled': False, 'minimum_independent_test_geometries': None, 'sample_size_justification': None, 'primary_estimand': 'scenario_success_per_scheduled_attempt', 'planning_notebook': '07_auditoria_diseno.ipynb', 'status': 'requires_separate_locked_protocol_and_independent_test_bank'}, 'report_revision': 'final_development_descriptive_v1', 'reference_selection_note': 'Gurobi elegido tras el diagnóstico de desarrollo: revisión descriptiva declarada, no hipótesis confirmatoria preespecificada. Se promedian todas las réplicas previstas.'}
print(json.dumps(COMPARISON_CONFIG, indent=2))


{
  "schema_version": "comparison-06-v2",
  "role": "development_only",
  "analysis_mode": "descriptive_only",
  "expected_methods": [
    "gurobi",
    "aco",
    "tabu",
    "am_supervised",
    "pomo",
    "hbg_facs",
    "edd",
    "ortools_routing",
    "pomo_untrained",
    "random_masked",
    "hbg_no_pheromone",
    "hbg_untrained"
  ],
  "reference_method": "gurobi",
  "metrics": [
    "scenario_strict_feasibility",
    "mean_travel_minutes",
    "cvar_travel_minutes",
    "mean_lateness_minutes",
    "mean_depot_overtime_minutes",
    "search_total_seconds"
  ],
  "accepted_bank": {
    "bank_id": "pilot00_280553a33c2ab594",
    "protocol_version": "pilot-00-v4",
    "manifest_sha256": "d8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02",
    "acceptance_status": "ready_for_method_development",
    "role": "development_only",
    "validation_run_id": "20260917T021417Z_665a3a34",
    "evidence": "docs/NOTEBOOK_00_VALIDATION.md"
  },
  "method_contract": {
    "sc

## 2. Drive y banco sellado
Se verifica el ID, el acta de aceptación y el SHA-256 del manifiesto antes de importar
el código del banco. No se selecciona automáticamente otro banco de Drive.
La aceptación de 00 habilita desarrollo; no prueba superioridad ni factibilidad
estocástica de las rutas exportadas por los métodos.


In [3]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/TESIS-EHBG-FACS')
import hashlib
def verify_before_import(folder):
    folder = Path(folder)
    manifest = json.loads((folder / 'CHECKSUMS.json').read_text())
    actual = {str(f.relative_to(folder)) for f in folder.rglob('*')
              if f.is_file() and f.name != 'CHECKSUMS.json' and '__pycache__' not in f.relative_to(folder).parts}
    if actual != set(manifest):
        raise ValueError('Inventario del banco distinto del manifiesto.')
    for relative, expected in manifest.items():
        file = folder / relative
        if not file.resolve().is_relative_to(folder.resolve()):
            raise ValueError('Ruta de manifiesto fuera del banco.')
        if hashlib.sha256(file.read_bytes()).hexdigest() != expected:
            raise ValueError(f'Hash incorrecto: {relative}')
    return manifest
if BANK_ID != BANK_REFERENCE['bank_id']:
    raise ValueError('BANK_ID difiere de la referencia validada; actualizar la referencia común antes de cambiar banco.')
remote_bank = DRIVE_ROOT / 'banks' / BANK_ID
manifest = verify_before_import(remote_bank)
manifest_sha = hashlib.sha256(json.dumps(manifest, sort_keys=True, separators=(',', ':'),
                                        ensure_ascii=False, allow_nan=False).encode('utf-8')).hexdigest()
if manifest_sha != BANK_REFERENCE['manifest_sha256']:
    raise ValueError('El banco no coincide con el manifiesto aceptado en 00; no importar ni ejecutar.')
acceptance = json.loads((remote_bank / 'acceptance.json').read_text())
if (acceptance.get('status') != BANK_REFERENCE['acceptance_status']
        or acceptance.get('protocol_version') != BANK_REFERENCE['protocol_version']):
    raise ValueError('El acta no coincide con la aceptación esperada del banco.')
BANK = WORK / 'banks' / BANK_ID
if not BANK.exists():
    BANK.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(remote_bank, BANK)
if verify_before_import(BANK) != manifest:
    raise ValueError('La copia local no coincide con Drive.')
shutil.copy2(BANK / 'code/svrplab_pilot.py', RUNTIME / 'svrplab/pilot.py')
print('Banco validado en 00, copiado y hashes verificados:', BANK_ID, manifest_sha)


Mounted at /content/drive
Banco validado en 00, copiado y hashes verificados: pilot00_280553a33c2ab594 d8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02


## 3. Código de comparación incluido
Estas celdas sólo escriben módulos. `pilot.py` se copia del banco verificado.
`search.py` y `experiments.py` aportan contratos y utilidades de archivos compartidas;
este notebook no invoca búsqueda ni evaluación de escenarios. Los módulos de
entrenamiento, pesos y datasets de profesores no se cargan.


In [4]:
%%writefile svrplab/__init__.py
"""Infraestructura experimental de la tesis HBG-FACS."""


Overwriting svrplab/__init__.py


In [5]:
%%writefile svrplab/search.py
"""Contrato de búsqueda determinista; evaluación estocástica en pilot, sin certificados.

El objetivo es el viaje con una cota uniforme de medias por arco. No representa
el costo óptimo del SVRP ni el costo completo de recurso todavía pendiente.
"""
from __future__ import annotations

import math
import numpy as np

from . import pilot as p

OBJECTIVE = 'conservative_upper_travel_minutes_v1'


class SearchProblem:
    def __init__(self, x, spec, time_scale=1000):
        self.x = x
        self.n = x.n
        self.vehicles = len(x.capacities)
        self.capacities = x.capacities
        self.demands = x.demands
        self.open = np.maximum(x.windows[:, 0], x.appear_times)
        self.close = x.windows[:, 1]
        self.horizon = float(x.windows[0, 1])
        self.distances = p.distances(x)
        self.travel = p.nominal_upper_matrix(x, spec, time_scale) / time_scale
        self.time_scale = time_scale

    def route_stats(self, route, vehicle):
        clock = travel = late = 0.0
        node = 0
        starts = []
        for customer in route:
            leg = float(self.travel[node, customer])
            travel += leg
            clock = max(clock + leg, float(self.open[customer]))
            late += max(0.0, clock - float(self.close[customer]))
            starts.append(clock)
            node = customer
        if route:
            leg = float(self.travel[node, 0])
            clock += leg
            travel += leg
        return {'travel_minutes': travel, 'lateness_minutes': late,
                'depot_overtime_minutes': max(0.0, clock - self.horizon),
                'capacity_excess': max(0.0, float(sum(self.demands[j] for j in route)
                                                - self.capacities[vehicle])),
                'service_starts': starts, 'return_minutes': clock}

    def score(self, routes):
        p.validate_routes(self.x, routes)
        stats = [self.route_stats(route, v) for v, route in enumerate(routes)]
        return (sum(r['lateness_minutes'] + r['depot_overtime_minutes'] for r in stats),
                sum(r['travel_minutes'] for r in stats))

    def result(self, routes, status, elapsed, history, **metadata):
        if not math.isfinite(elapsed) or elapsed < 0:
            raise ValueError('Duración de búsqueda inválida.')
        if routes is not None:
            routes = [[int(j) for j in route] for route in routes]
            violation, cost = self.score(routes)
            metadata.update(search_violation_minutes=violation,
                            search_objective_minutes=cost,
                            conservative_feasible=violation <= 1e-9)
        metadata['objective_scope'] = OBJECTIVE
        return {'routes': routes, 'status': status, 'solve_seconds': float(elapsed),
                'history': history, 'metadata': metadata}


Writing svrplab/search.py


In [6]:
%%writefile svrplab/experiments.py
"""Ejecución Colab, evaluación única y checkpoints inmutables para métodos 01–06.

No modifica el banco ni entrega certificados a los algoritmos. Cada resultado
se reevalúa desde sus rutas persistidas con el pilot.py guardado en el banco.
"""
from __future__ import annotations

import csv
import json
import math
import shutil
import time
import uuid
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from . import pilot as p
from .search import OBJECTIVE, SearchProblem

SCHEMA = 'methods-v2'
SUMMARY_METRICS = [
    'scenarios', 'alpha', 'mean_travel_minutes', 'cvar_travel_minutes',
    'cost_variance_population', 'std_travel_minutes', 'scenario_strict_feasibility',
    'mean_customer_violation_fraction', 'mean_lateness_minutes',
    'mean_depot_overtime_minutes', 'mean_waiting_minutes', 'mean_base_minutes',
    'mean_net_execution_delta_minutes', 'mean_late_customers', 'mean_recourse_reorders',
    'used_vehicles', 'evaluation_seconds', 'trace_validation_seconds',
]


def write_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(fields or dict.fromkeys(k for row in rows for k in row))
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def native(value):
    """Normaliza escalares del evaluador sellado sin modificar su implementación."""
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return [native(item) for item in value.tolist()]
    if isinstance(value, dict):
        return {key: native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(item) for item in value]
    return value


def write_json(path, value):
    p.write_json(path, native(value))


def read_json(path):
    return json.loads(Path(path).read_text())


def validate_scenarios(scenarios, config):
    if set(scenarios) != set(config['scenarios']):
        raise ValueError('Splits del banco incompatibles.')
    seen = set()
    for split, count in config['scenarios'].items():
        seeds = scenarios[split]
        if (len(seeds) != count or len(set(seeds)) != count
                or any(isinstance(s, bool) or not isinstance(s, int) or s < 0 for s in seeds)
                or seen.intersection(seeds)):
            raise ValueError('Escenarios repetidos, solapados o con tamaño/tipo incorrecto.')
        seen.update(seeds)


def validate_config(config):
    if (config['schema_version'] != SCHEMA or config['role'] != 'development_only'
            or config['objective'] != OBJECTIVE or config['scenario_split'] != 'evaluation'
            or config['export_all_scenario_events'] is not True):
        raise ValueError('Contrato experimental incompatible.')
    if not math.isfinite(config['search_seconds']) or config['search_seconds'] <= 0:
        raise ValueError('Presupuesto debe ser positivo y finito.')
    seeds = config['seeds']
    if not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s < 0 for s in seeds):
        raise ValueError('Semillas de métodos inválidas.')
    if type(config['time_scale']) is not int or config['time_scale'] < 1:
        raise ValueError('Escala de tiempo inválida.')


def route_rows(x, routes, context):
    rows = []
    for vehicle, route in enumerate(routes):
        load = 0.0
        for position, node in enumerate([0, *route, 0]):
            load += float(x.demands[node])
            rows.append({**context, 'vehicle': vehicle, 'position': position,
                         'node': node, 'empty_vehicle': not bool(route),
                         'x': float(x.locations[node, 0]), 'y': float(x.locations[node, 1]),
                         'demand': float(x.demands[node]), 'cumulative_load': load,
                         'capacity': float(x.capacities[vehicle]),
                         'window_open': float(x.windows[node, 0]),
                         'window_close': float(x.windows[node, 1]),
                         'appear_time': float(x.appear_times[node])})
    return rows


def validate_trace(x, routes, trace, raw):
    for key, value in raw.items():
        actual = trace[key]
        if isinstance(value, float):
            if not math.isclose(actual, value, rel_tol=1e-12, abs_tol=1e-9):
                raise AssertionError(f'Traza distinta del evaluador: {key}')
        elif actual != value:
            raise AssertionError(f'Traza distinta del evaluador: {key}')
    p.validate_routes(x, trace['realized_routes'])
    if any(sorted(a) != sorted(b) for a, b in zip(routes, trace['realized_routes'])):
        raise AssertionError('El recurso cambió la asignación de vehículo.')
    events = trace['events']
    expected = x.n + sum(bool(r) for r in routes)
    if len(events) != expected:
        raise AssertionError('Eventos incompletos.')
    if any(type(e['vehicle']) is not int or not 0 <= e['vehicle'] < len(routes) for e in events):
        raise AssertionError('Vehículo inválido en eventos.')
    for vehicle, realized in enumerate(trace['realized_routes']):
        vehicle_events = [e for e in events if e['vehicle'] == vehicle]
        path = [0, *realized, 0] if realized else [0]
        if [(e['from'], e['to']) for e in vehicle_events] != list(zip(path, path[1:])):
            raise AssertionError('Arcos distintos de la ruta realizada.')
        previous = 0.0
        for event in vehicle_events:
            node = event['to']
            start = max(event['arrival'], x.windows[node, 0], x.appear_times[node]) if node else event['arrival']
            expected_late = max(0.0, start - x.windows[node, 1])
            if (not all(math.isfinite(event[k]) for k in ('departure', 'arrival', 'service_start', 'late'))
                    or not math.isclose(event['departure'], previous, abs_tol=1e-9)
                    or event['arrival'] < event['departure'] - 1e-9
                    or not math.isclose(event['service_start'], start, abs_tol=1e-9)
                    or not math.isclose(event['late'], expected_late, abs_tol=1e-9)):
                raise AssertionError('Horario de eventos incoherente.')
            previous = event['service_start']
    travel = sum(e['arrival'] - e['departure'] for e in events)
    waiting = sum(e['service_start'] - e['arrival'] for e in events)
    late = sum(e['late'] for e in events if e['to'] != 0)
    overtime = sum(e['late'] for e in events if e['to'] == 0)
    for value, key in [(travel, 'travel_minutes'), (waiting, 'waiting_minutes'),
                       (late, 'lateness_minutes'), (overtime, 'depot_overtime_minutes')]:
        if not math.isclose(value, raw[key], rel_tol=1e-10, abs_tol=1e-7):
            raise AssertionError(f'Eventos no reconcilian {key}.')


def evaluate_solution(x, solution, seeds, spec, policy, alpha, directory, context):
    """Nunca selecciona/tunea con evaluation; puntúa una solución ya congelada."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / 'environment.json', p.environment_report())
    routes = solution['routes']
    before = x.sha
    p.validate_routes(x, routes)
    summary, records = p.evaluate(x, routes, seeds, spec, policy, alpha)
    if [r['scenario_seed'] for r in records] != seeds:
        raise AssertionError('El evaluador cambió el conjunto u orden de escenarios.')
    trace_started = time.perf_counter()
    event_count = route_count = 0
    # Streaming: no acumular millones de eventos en RAM para n=300.
    with (directory / 'events.csv').open('w', newline='', encoding='utf-8') as ef, \
            (directory / 'realized_routes.csv').open('w', newline='', encoding='utf-8') as rf:
        ew = rw = None
        for record in records:
            seed = record['scenario_seed']
            trace = p.simulate(x, routes, seed, spec, policy, trace=True)
            validate_trace(x, routes, trace, record)
            trace_context = {**context, 'policy': policy, 'scenario_seed': str(seed)}
            for event_index, event in enumerate(trace['events']):
                row = {**trace_context, 'event_index': event_index, **event}
                if ew is None:
                    ew = csv.DictWriter(ef, fieldnames=list(row)); ew.writeheader()
                ew.writerow(row)
                event_count += 1
            for row in route_rows(x, trace['realized_routes'], trace_context):
                if rw is None:
                    rw = csv.DictWriter(rf, fieldnames=list(row)); rw.writeheader()
                rw.writerow(row)
                route_count += 1
    summary['trace_validation_seconds'] = time.perf_counter() - trace_started
    costs = [r['travel_minutes'] for r in records]
    for source in ['waiting_minutes', 'base_minutes', 'net_execution_delta_minutes',
                   'late_customers', 'recourse_reorders']:
        summary['mean_' + source] = float(np.mean([r[source] for r in records]))
    summary['std_travel_minutes'] = float(np.std(costs, ddof=0))
    # CVaR por integración de masa en la cola: control independiente de la fórmula de pilot.
    descending = sorted(costs, reverse=True)
    mass = len(costs) * (1 - alpha)
    whole = min(len(costs), int(math.floor(mass)))
    tail_sum = sum(descending[:whole])
    if whole < len(costs):
        tail_sum += (mass - whole) * descending[whole]
    if not math.isclose(tail_sum / mass, summary['cvar_travel_minutes'], rel_tol=1e-10):
        raise AssertionError('CVaR no coincide con integración de la cola.')
    if x.sha != before:
        raise AssertionError('La instancia cambió durante evaluación.')
    raw_rows = [{**context, 'policy': policy, **r, 'scenario_seed': str(r['scenario_seed'])}
                for r in records]
    write_csv(directory / 'scenario_metrics.csv', raw_rows)
    write_json(directory / 'summary.json', {**context, **summary})
    qc = {'status': 'passed', 'structural_valid': True, 'scenarios': len(records),
          'event_rows': event_count, 'realized_route_rows': route_count,
          'expected_event_rows': len(seeds) * (x.n + solution['nominal']['used_vehicles']),
          'expected_realized_route_rows': len(seeds) * (x.n + 2 * len(x.capacities)),
          'instance_unchanged': x.sha == before, 'trace_matches_evaluator': True,
          'recourse_preserves_vehicle_assignment': True, 'cvar_independently_checked': True,
          'nominal_feasible': solution['nominal']['strict_feasible'],
          'stochastic_feasibility_is_a_measurement': True}
    if qc['event_rows'] != qc['expected_event_rows'] or qc['realized_route_rows'] != qc['expected_realized_route_rows']:
        raise AssertionError('Número de filas de exportación incorrecto.')
    # Releer CSVs comprueba la persistencia, las semillas de 64 bits y el número de filas.
    for name, expected_rows in [('events.csv', event_count), ('realized_routes.csv', route_count),
                                ('scenario_metrics.csv', len(seeds))]:
        with (directory / name).open(newline='', encoding='utf-8') as stream:
            rows = csv.DictReader(stream)
            count = 0
            observed = set()
            for row in rows:
                observed.add(int(row['scenario_seed'])); count += 1
            if count != expected_rows or observed != set(seeds):
                raise AssertionError(f'Exportación incompleta: {name}')
    write_json(directory / 'validation.json', qc)
    return {**context, **summary}


def save_plot(x, routes, path, title):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 6))
    for vehicle, route in enumerate(routes):
        if not route:
            continue
        coords = x.locations[[0, *route, 0]]
        ax.plot(coords[:, 0], coords[:, 1], '-o', markersize=2, linewidth=.8,
                label=f'V{vehicle}')
    ax.scatter(*x.locations[0], marker='*', s=160, color='black', zorder=5)
    ax.set(title=title, xlabel='x (unidad fuente)', ylabel='y (unidad fuente)', aspect='equal')
    if sum(bool(r) for r in routes) <= 15:
        ax.legend(fontsize=6, ncol=3)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def _commit(directory, remote):
    p.seal_bundle(directory)
    _sync(directory, remote)


def _sync(local, remote):
    """Conservar staging interrumpido; promoverlo sólo si su hash es idéntico."""
    staging = remote.with_name(remote.name + '.partial')
    if not remote.exists() and staging.exists():
        try:
            complete = p.verify_bundle(staging) == p.verify_bundle(local)
        except (ValueError, OSError, json.JSONDecodeError):
            complete = False
        if complete:
            staging.rename(remote)
        else:
            staging.rename(staging.with_name(staging.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    p.sync_verified(local, remote)


def _recover(local, remote):
    if (local / 'CHECKSUMS.json').exists():
        p.verify_bundle(local)
        _sync(local, remote)
        return True
    if remote.exists():
        # Conservar la escritura local incompleta, sin sobrescribirla ni mezclarla.
        if local.exists():
            local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
        _sync(remote, local)
        return True
    if local.exists():
        local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    return False


def comparison_protocol(bank_hash, evaluator_hash, bank_config, config):
    """Mismo problema/evaluación/presupuesto; algoritmos y entrenamiento pueden diferir."""
    return {'schema_version': SCHEMA, 'bank_manifest_sha256': bank_hash,
            'evaluator_sha256': evaluator_hash,
            'runner_sha256': p.file_hash(__file__),
            'search_contract_sha256': p.file_hash(Path(__file__).with_name('search.py')),
            'objective': config['objective'], 'time_scale': config['time_scale'],
            'role': config['role'], 'solver_seeds': config['seeds'],
            'search_seconds': config['search_seconds'], 'scenario_split': config['scenario_split'],
            'policies': bank_config['policies'], 'alpha': bank_config['alpha_cvar'],
            'export_all_scenario_events': config['export_all_scenario_events']}


def provenance_columns(provenance):
    fields = ['training_id', 'model_checkpoint_sha256', 'fold_id', 'training_seed',
              'train_geometry_groups', 'validation_geometry_groups', 'heldout_geometry_groups',
              'training_seconds', 'validation_seconds', 'teacher_seconds', 'offline_cost_id',
              'offline_manifest_sha256', 'checkpoint_selection', 'training_scope']
    result = {key: provenance.get(key) for key in fields}
    for key in ['training_seconds', 'validation_seconds', 'teacher_seconds']:
        result[key] = provenance.get(key, 0.0)
        if isinstance(result[key], bool) or not isinstance(result[key], (float, int)) or not math.isfinite(result[key]) or result[key] < 0:
            raise ValueError(f'Costo offline inválido: {key}')
    for key, value in result.items():
        if isinstance(value, (dict, list, tuple)):
            result[key] = json.dumps(value, sort_keys=True, separators=(',', ':'))
    return result


def validate_learned_provenance(provenance, x, seed):
    groups = [provenance.get(role) for role in
              ['train_geometry_groups', 'validation_geometry_groups', 'heldout_geometry_groups']]
    if any(not isinstance(group, list) or not group for group in groups):
        raise ValueError('Faltan grupos de entrenamiento/validación/heldout del modelo.')
    train, validation, heldout = map(set, groups)
    if (train & validation or train & heldout or validation & heldout
            or x.provenance['geometry_group_sha256'] not in heldout
            or provenance.get('training_seed') != seed):
        raise ValueError('Procedencia del modelo tiene fuga de geometría o semilla distinta.')
    checkpoint = provenance.get('model_checkpoint_sha256', '')
    if not provenance.get('training_id') or len(checkpoint) != 64 or any(c not in '0123456789abcdef' for c in checkpoint):
        raise ValueError('Falta identidad/hash de checkpoint del modelo aprendido.')


def run_methods(bank, config, solvers, work_root, drive_root, run_id=None, validation_artifacts=(),
                method_availability=None, method_provenance=None):
    """solvers: {method: callable(problem, seed, config, output_dir)}.

    Rerun con run_id explícito recupera checkpoints; sin ID crea otra réplica
    trazable. La comparación usa comparison_id, sin confundir réplicas con N.
    """
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Experimentación exclusivamente en Google Colab.') from exc
    validate_config(config)
    bank = Path(bank)
    instances = sorted(p.load_bank(bank), key=lambda x: x.n)
    bank_config = read_json(bank / 'config.json')
    bank_hash = p.verify_bundle(bank)
    spec = p.Traffic(**bank_config['traffic'])
    nominal_spec = p.Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
    scenarios = {x.n: read_json(bank / f'scenarios/n{x.n:04d}.json') for x in instances}
    for value in scenarios.values():
        validate_scenarios(value, bank_config)
    if not solvers or not set(solvers).issubset(config['methods']):
        raise ValueError('Métodos solicitados incompatibles con configuración.')
    method_availability = method_availability or {name: 'enabled' for name in solvers}
    if set(method_availability) != set(solvers) or not set(method_availability.values()).issubset({'enabled', 'license_pending'}):
        raise ValueError('Disponibilidad de métodos inválida.')
    package = Path(__file__).parent
    source_hashes = {f.name: p.file_hash(f) for f in sorted(package.glob('*.py'))}
    method_provenance = method_provenance or {}
    protocol = comparison_protocol(bank_hash, p.file_hash(p.__file__), bank_config, config)
    comparison_id = p.digest(protocol)
    signature = p.digest({'protocol': protocol, 'config': config,
                          'methods': sorted(solvers), 'source_hashes': source_hashes,
                          'method_availability': method_availability, 'method_provenance': method_provenance})
    run_id = run_id or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_') + uuid.uuid4().hex[:8]
    if Path(run_id).name != run_id or run_id in ('.', '..'):
        raise ValueError('run_id debe ser un nombre de carpeta simple.')
    root = Path(work_root) / 'method_runs' / bank.name / run_id
    remote = Path(drive_root) / 'method_runs' / bank.name / run_id
    print(f'RUN_ID = {run_id}\nDrive: {remote}', flush=True)
    if _recover(root / 'meta', remote / 'meta'):
        if read_json(root / 'meta/plan.json')['signature'] != signature:
            raise ValueError('No se puede reanudar con otro banco/código/protocolo/configuración/disponibilidad de licencia. Use RUN_ID nuevo.')
    else:
        meta = root / 'meta'
        meta.mkdir(parents=True)
        write_json(meta / 'plan.json', {'run_id': run_id, 'signature': signature,
                     'comparison_id': comparison_id, 'protocol': protocol, 'config': config,
                     'methods': sorted(solvers), 'source_hashes': source_hashes,
                     'method_availability': method_availability,
                     'method_provenance': method_provenance,
                     'bank_id': bank.name, 'role': 'development_only',
                     'created_utc': datetime.now(timezone.utc).isoformat()})
        write_json(meta / 'environment.json', p.environment_report())
        for source in package.glob('*.py'):
            destination = meta / 'code' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        for source in map(Path, validation_artifacts):
            destination = meta / 'validation' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        write_json(meta / 'metrics_schema.json', metric_schema())
        _commit(meta, remote / 'meta')
    if _recover(root / 'report', remote / 'report'):
        for relative, expected in read_json(root / 'report/bundles.json').items():
            if p.verify_bundle(remote / relative) != expected:
                raise ValueError(f'Checkpoint remoto modificado: {relative}')
            if not _recover(root / relative, remote / relative):
                raise ValueError(f'Checkpoint ausente: {relative}')
        print('Reporte completo recuperado y verificado.', flush=True)
        return root
    summaries, failures = [], []
    for x in instances:
        for method, solver in solvers.items():
            for seed in config['seeds']:
                key = f'{method}/n{x.n:04d}/seed_{seed}'
                job, remote_job = root / key, remote / key
                learned = method_provenance.get(method, {}).get(f'{x.sha}:{seed}', {})
                if method in ('am_supervised', 'pomo', 'hbg_facs', 'hbg_no_pheromone') and not learned:
                    raise ValueError('Falta procedencia del modelo entrenado para este intento.')
                if method in ('am_supervised', 'pomo', 'hbg_facs', 'hbg_no_pheromone'):
                    validate_learned_provenance(learned, x, seed)
                context = {'schema_version': SCHEMA, 'comparison_id': comparison_id,
                    'run_id': run_id, 'bank_id': bank.name, 'bank_manifest_sha256': bank_hash,
                    'evaluator_sha256': p.file_hash(p.__file__), 'method': method,
                    'solver_seed': seed, 'instance_id': x.instance_id, 'instance_sha256': x.sha,
                    'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                    'n_customers': x.n, 'objective_scope': OBJECTIVE,
                    'search_budget_seconds': config['search_seconds'],
                    'scenario_set_sha256': p.digest(scenarios[x.n]['evaluation']),
                    **provenance_columns(learned)}
                print(f'{key}: búsqueda / recuperación...', flush=True)
                solution_dir = job / 'solution'
                if not _recover(solution_dir, remote_job / 'solution'):
                    solution_dir.mkdir(parents=True)
                    started = time.perf_counter()
                    try:
                        problem = SearchProblem(x, spec, config['time_scale'])
                        preparation = time.perf_counter() - started
                        settings = {**config['methods'][method],
                                    'seconds': max(0.0, config['search_seconds'] - preparation)}
                        if settings['seconds'] <= 0:
                            solution = problem.result(None, 'time_limit_no_solution', preparation, [])
                        else:
                            solution = solver(problem, seed, settings, solution_dir)
                        solution['search_total_seconds'] = time.perf_counter() - started
                        solution['preparation_seconds'] = preparation
                        solution['nominal'] = None
                        if solution['routes'] is not None:
                            p.validate_routes(x, solution['routes'])
                            conservative = problem.score(solution['routes'])
                            solution['metadata'].update(search_violation_minutes=conservative[0],
                                search_objective_minutes=conservative[1],
                                conservative_feasible=conservative[0] <= 1e-9)
                            solution['nominal'] = p.simulate(x, solution['routes'], 0, nominal_spec,
                                                             'fixed_order', trace=True)
                            if conservative[0] <= 1e-9 and not solution['nominal']['strict_feasible']:
                                raise AssertionError('Solución conservadora viola el simulador nominal.')
                    except Exception as exc:
                        solution = {'routes': None, 'nominal': None, 'status': 'solver_error',
                                    'solve_seconds': time.perf_counter() - started,
                                    'search_total_seconds': time.perf_counter() - started,
                                    'history': [], 'metadata': {'error_type': type(exc).__name__,
                                    'error': str(exc), 'objective_scope': OBJECTIVE}}
                    write_json(solution_dir / 'solution.json', solution)
                    write_json(solution_dir / 'environment.json', p.environment_report())
                    # Read-back antes de evaluar: la solución persistida es la autoridad.
                    solution = read_json(solution_dir / 'solution.json')
                    write_json(solution_dir / 'identity.json', context)
                    write_csv(solution_dir / 'search_history.csv', solution['history'],
                              None if solution['history'] else ['iteration', 'elapsed_seconds'])
                    if solution['routes'] is not None:
                        write_csv(solution_dir / 'planned_routes.csv', route_rows(x, solution['routes'], context))
                        write_csv(solution_dir / 'nominal_events.csv',
                                  [{**context, **e} for e in solution['nominal']['events']])
                        save_plot(x, solution['routes'], solution_dir / 'routes.png',
                                  f'{method} · n={x.n} · seed={seed} · {solution["status"]}')
                    _commit(solution_dir, remote_job / 'solution')
                solution = read_json(solution_dir / 'solution.json')
                if read_json(solution_dir / 'identity.json') != context:
                    raise ValueError('Checkpoint de solución corresponde a otro intento.')
                meta = solution['metadata']
                base = {**context, 'solver_status': solution['status'],
                        'structural_valid': solution['routes'] is not None,
                        'nominal_feasible': solution['nominal']['strict_feasible'] if solution['nominal'] else None,
                        'conservative_feasible': meta.get('conservative_feasible'),
                        'search_violation_minutes': meta.get('search_violation_minutes'),
                        'search_objective_minutes': meta.get('search_objective_minutes'),
                        'solve_seconds': solution['solve_seconds'],
                        'search_total_seconds': solution['search_total_seconds'],
                        'best_bound': meta.get('best_bound'), 'relative_gap': meta.get('relative_gap'),
                        'surrogate_optimal': meta.get('surrogate_optimal'),
                        'model_load_seconds': meta.get('model_load_seconds', 0.0),
                        'tuning_seconds_this_run': 0.0}
                for policy in bank_config['policies']:
                    result_dir = job / policy
                    if solution['routes'] is None:
                        summary = {**base, 'policy': policy, 'evaluation_status': 'no_solution',
                                   **dict.fromkeys(SUMMARY_METRICS)}
                        failures.append({**context, 'policy': policy, 'status': solution['status'],
                                         'error': meta.get('error', '')})
                    else:
                        print(f'{key}: {policy}, {len(scenarios[x.n]["evaluation"])} escenarios...', flush=True)
                        if not _recover(result_dir, remote_job / policy):
                            evaluate_solution(x, solution, scenarios[x.n]['evaluation'], spec, policy,
                                              bank_config['alpha_cvar'], result_dir, context)
                            _commit(result_dir, remote_job / policy)
                        evaluated = read_json(result_dir / 'summary.json')
                        if any(evaluated.get(k) != v for k, v in context.items()) or evaluated.get('policy') != policy:
                            raise ValueError('Checkpoint de evaluación corresponde a otro intento o política.')
                        summary = {**base, **evaluated,
                                   'evaluation_status': 'validated'}
                    summaries.append(summary)
    if p.verify_bundle(bank) != bank_hash:
        raise AssertionError('El banco fue modificado.')
    report = root / 'report'
    report.mkdir(parents=True, exist_ok=True)
    write_csv(report / 'summary.csv', summaries)
    write_csv(report / 'failures.csv', failures, None if failures else ['method', 'n_customers', 'solver_seed', 'status'])
    write_json(report / 'summary.json', summaries)
    inventory = {'meta': p.verify_bundle(root / 'meta')}
    for x in instances:
        for method in solvers:
            for seed in config['seeds']:
                key = f'{method}/n{x.n:04d}/seed_{seed}'
                for part in ['solution', *bank_config['policies']]:
                    relative = f'{key}/{part}'
                    if (root / relative / 'CHECKSUMS.json').exists():
                        inventory[relative] = p.verify_bundle(root / relative)
                        if p.verify_bundle(remote / relative) != inventory[relative]:
                            raise ValueError(f'Copia Drive distinta: {relative}')
    write_json(report / 'bundles.json', inventory)
    save_summary_plot(summaries, report / 'metrics.png', bank_config['policies'])
    write_json(report / 'validation.json', {'status': 'completed', 'bank_unchanged': True,
        'attempts': len(instances) * len(solvers) * len(config['seeds']),
        'summary_rows': len(summaries), 'unsolved_policy_rows': len(failures),
        'not_a_claim_of_all_routes_feasible': True, 'ready_for_confirmatory_comparison': False})
    _commit(report, remote / 'report')
    print(f'COMPLETADO: exportaciones verificadas en {remote}', flush=True)
    return root


def save_summary_plot(summaries, path, policies):
    import matplotlib.pyplot as plt
    rows = [r for r in summaries if r['evaluation_status'] == 'validated']
    if not rows:
        return
    fig, axes = plt.subplots(len(policies), 2, figsize=(12, 4 * len(policies)), squeeze=False)
    for row, policy in enumerate(policies):
        for method in sorted({r['method'] for r in rows}):
            data = [r for r in rows if r['policy'] == policy and r['method'] == method]
            for column, metric in enumerate(['scenario_strict_feasibility', 'mean_travel_minutes']):
                axes[row, column].scatter([r['n_customers'] for r in data],
                                         [r[metric] for r in data], label=method, alpha=.65)
                axes[row, column].set(xlabel='Clientes (una geometría por tamaño)', ylabel=metric,
                                      title=f'{policy} · todas las semillas')
                axes[row, column].legend()
        axes[row, 0].set_ylim(-.03, 1.03)
    fig.suptitle('Piloto de desarrollo · costo de viaje y factibilidad deben interpretarse juntos')
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def metric_schema():
    return {'schema_version': SCHEMA, 'objective_scope': OBJECTIVE,
            'cost_scope': 'travel_only_not_full_recourse_cost',
            'scenario_strict_feasibility': 'Fracción de escenarios con cobertura, capacidad, ventanas y retorno válidos.',
            'cvar_travel_minutes': 'CVaR empírico con masa fraccionaria, alpha del banco.',
            'cost_variance_population': 'Varianza de viaje entre escenarios, ddof=0.',
            'mean_net_execution_delta_minutes': 'Viaje ejecutado menos viaje base planeado; puede ser negativo; no es Q óptimo.',
            'relative_gap': 'Gap Gurobi de la misma formulación conservadora; null si no disponible.',
            'search_total_seconds': 'Preparación y llamada al método; incluye construcción, extracción y artefactos propios del solver.',
            'evaluation_seconds': 'Tiempo informado por el evaluador común para Monte Carlo.',
            'trace_validation_seconds': 'Segunda pasada determinista, validación y escritura de trazas.',
            'missing_metrics': 'null en JSON / vacío en CSV cuando no hay solución; no convertir en cero.',
            'scenario_seed': 'Entero de 64 bits serializado en decimal: leer CSV como texto para evitar redondeo.',
            'independent_unit': 'geometry_group_sha256; semillas y escenarios no aumentan N.',
            'training_seconds': 'Costo total del modelo de ese fold/semilla, repetido por política; no sumar filas.',
            'teacher_seconds': 'Etiquetado compartido de geometrías train/validation; costos únicos en manifiesto offline.',
            'offline_cost_id': 'Identifica el inventario de costos únicos de entrenamiento/etiquetado.',
            'comparison_id': 'Mismo banco/evaluador/runner/objetivo/presupuesto/escenarios; no exige algoritmos iguales.',
            'solver_seed': 'En am_supervised/pomo/hbg_facs identifica entrenamiento y búsqueda; no equivale al mismo flujo RNG de ACO.',
            'comparison_keys': ['comparison_id', 'instance_sha256', 'solver_seed', 'policy'],
            'summary_metrics': SUMMARY_METRICS}


Writing svrplab/experiments.py


In [7]:
%%writefile svrplab/comparison_stats.py
"""06: descripción del piloto por geometría; ejecución exclusivamente en Colab.

No calcula p-valores, intervalos poblacionales, bootstrap ni rankings globales.
Las semillas de solver no se emparejan como RNG equivalentes. Los escenarios
CRN se emparejan sólo después de promediar las réplicas de cada método y sólo
para cohortes completas. CVaR se promedia por plan; nunca se agrupan los costos
de planes distintos para fabricar un CVaR diferente.

El loader de 06 verifica procedencia, sellos, protocolo y evaluaciones; este
módulo verifica claves, cobertura y agregación. Una geometría sigue contando
una vez aunque tenga varias instancias, ventanas, réplicas o escenarios.
"""
from __future__ import annotations

from collections import defaultdict
from collections.abc import Mapping
import hashlib
import math
from pathlib import Path
import re
from statistics import fmean


SCHEMA = 'comparison-06-v2'
BASE_METHODS = ['gurobi', 'aco', 'tabu', 'am_supervised', 'pomo', 'hbg_facs']
CONTROLS = ['edd', 'ortools_routing', 'pomo_untrained', 'random_masked', 'hbg_no_pheromone', 'hbg_untrained']
METHODS = BASE_METHODS + CONTROLS
CONTROLLED_PAIRS = [('pomo', 'pomo_untrained'), ('pomo', 'random_masked'),
                    ('hbg_facs', 'hbg_no_pheromone'), ('hbg_facs', 'hbg_untrained')]
METRICS = ['scenario_strict_feasibility', 'mean_travel_minutes', 'cvar_travel_minutes',
           'mean_lateness_minutes', 'mean_depot_overtime_minutes', 'search_total_seconds']
SCENARIO_METRICS = {'scenario_strict_feasibility': 'strict_feasible',
                    'mean_travel_minutes': 'travel_minutes',
                    'mean_lateness_minutes': 'lateness_minutes',
                    'mean_depot_overtime_minutes': 'depot_overtime_minutes'}
_BOUND_REL_TOL, _BOUND_ABS_TOL = 1e-8, 1e-6


def _number(value, *, boolean=False):
    if value is None or value == '':
        return None
    if isinstance(value, bool):
        if boolean:
            return float(value)
        raise ValueError('Un booleano no representa una métrica numérica.')
    if boolean and value in ('True', 'False', 'true', 'false'):
        return float(value in ('True', 'true'))
    try:
        result = float(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError('Métrica no numérica.') from error
    if not math.isfinite(result):
        raise ValueError('Una medición disponible debe ser finita; ausente se representa por null.')
    return result


def _seed(value):
    # Nunca pasar las semillas de 64 bits por float/pandas inferencia numérica.
    if type(value) is int and value >= 0:
        return value
    if isinstance(value, str) and re.fullmatch(r'[0-9]+', value):
        return int(value)
    raise ValueError('Las semillas deben ser enteros exactos o cadenas decimales no negativas.')


def _config(config):
    if (not isinstance(config, Mapping) or config.get('schema_version') != SCHEMA
            or config.get('role') != 'development_only'
            or config.get('analysis_mode') != 'descriptive_only'):
        raise ValueError('06 sólo admite comparación descriptiva de desarrollo.')
    methods, metrics = config.get('expected_methods'), config.get('metrics')
    if (not isinstance(methods, list) or len(methods) != len(set(methods))
            or not set(BASE_METHODS) <= set(methods) or not set(methods) <= set(METHODS)
            or config.get('reference_method') not in ('aco', 'gurobi')):
        raise ValueError('Se requieren los seis métodos, controles reconocidos y referencia descriptiva ACO histórica o Gurobi declarada.')
    if not isinstance(metrics, list) or len(metrics) != len(METRICS) or set(metrics) != set(METRICS):
        raise ValueError('Las métricas descriptivas deben coincidir con el protocolo 06.')
    if (config.get('ready_for_confirmatory', False) is not False
            or config.get('future_confirmatory_design', {}).get('enabled', False) is not False):
        raise ValueError('El piloto no permite habilitar inferencia confirmatoria.')
    return list(methods), list(metrics)


def _identity(registry):
    result = {}
    for row in registry:
        if not isinstance(row, Mapping):
            raise ValueError('Registro de instancias inválido.')
        for key in ('instance_id', 'instance_sha256', 'geometry_group_sha256'):
            if not isinstance(row.get(key), str) or not row[key]:
                raise ValueError(f'Falta identidad: {key}.')
        if type(row.get('n_customers')) is not int or row['n_customers'] < 1:
            raise ValueError('n_customers debe ser entero positivo.')
        sha = row['instance_sha256']
        if sha in result:
            raise ValueError('Instancia duplicada en registry.')
        result[sha] = dict(row)
    if not result:
        raise ValueError('El registro de instancias no puede estar vacío.')
    return result


def _row_key(row, registry, policies, methods, seeds):
    if not isinstance(row, Mapping):
        raise ValueError('Fila de resultados inválida.')
    sha, policy, method = row.get('instance_sha256'), row.get('policy'), row.get('method')
    seed = _seed(row.get('solver_seed'))
    if sha not in registry or policy not in policies or method not in methods or seed not in seeds:
        raise ValueError('Fila ajena a la cohorte preespecificada.')
    identity = registry[sha]
    for key in ('instance_id', 'geometry_group_sha256', 'n_customers'):
        if key in row and row[key] != identity[key]:
            raise ValueError(f'Identidad de fila incompatible: {key}.')
    return sha, policy, method, seed


def _category(row):
    if row is None:
        return 'missing'
    if row.get('solver_status') == 'license_pending':
        return 'license_pending'
    if row.get('evaluation_status') == 'validated':
        return 'evaluated'
    return 'failed'


def _context(identity, policy, method):
    return {key: identity[key] for key in ('instance_id', 'instance_sha256', 'geometry_group_sha256',
                                          'n_customers')} | {'policy': policy, 'method': method}


def _case_tables(index, registry, policies, seeds, methods, metrics):
    output = []
    for sha, identity in registry.items():
        for policy in policies:
            for method in methods:
                rows = [index.get((sha, policy, method, seed)) for seed in seeds]
                counts = {name: sum(_category(row) == name for row in rows)
                          for name in ('evaluated', 'failed', 'license_pending', 'missing')}
                for metric in metrics:
                    runtime = metric == 'search_total_seconds'
                    feasible_cost = metric in ('mean_travel_minutes', 'cvar_travel_minutes')
                    eligible = [row for row in rows if row is not None and (runtime or (
                        _category(row) == 'evaluated' and (not feasible_cost or (
                            row.get('conservative_feasible') is True and row.get('structural_valid') is True))))]
                    values = [_number(row.get(metric)) for row in eligible]
                    values = [value for value in values if value is not None]
                    if any(value < 0 for value in values):
                        raise ValueError(f'Una métrica no negativa contiene un valor negativo: {metric}.')
                    if metric == 'scenario_strict_feasibility' and any(value > 1 for value in values):
                        raise ValueError('La fracción de factibilidad debe pertenecer a [0,1].')
                    output.append(_context(identity, policy, method) | {
                        'metric': metric, 'mean_conditional': fmean(values) if values else None,
                        'expected': len(seeds), **counts, 'metric_available': len(values),
                        'evaluated_fraction': counts['evaluated'] / len(seeds),
                        'metric_coverage_fraction': len(values) / len(seeds),
                        'cohort_complete': counts['evaluated'] == len(seeds) and len(values) == len(seeds),
                        'conditioning': ('recorded_attempts_with_runtime' if runtime else
                                         'validated_conservative_feasible_plans' if feasible_cost else
                                         'validated_solutions_with_metric'),
                        'aggregation': 'mean_over_available_expected_solver_replicas',
                        'cvar_scope': 'mean_of_per_plan_empirical_cvar' if metric == 'cvar_travel_minutes' else None,
                    })
    return output


def _geometry_tables(cases):
    groups = defaultdict(list)
    for row in cases:
        groups[(row['geometry_group_sha256'], row['policy'], row['method'], row['metric'])].append(row)
    output = []
    for (geometry, policy, method, metric), rows in groups.items():
        values = [row['mean_conditional'] for row in rows if row['mean_conditional'] is not None]
        counts = {name: sum(row[name] for row in rows)
                  for name in ('expected', 'evaluated', 'failed', 'license_pending', 'missing', 'metric_available')}
        output.append({
            'geometry_group_sha256': geometry, 'policy': policy, 'method': method, 'metric': metric,
            'n_customers': '|'.join(str(n) for n in sorted({row['n_customers'] for row in rows})),
            'expected_instances': len(rows), 'instances_with_measurement': len(values),
            'fully_evaluated_instances': sum(row['cohort_complete'] for row in rows),
            'mean_conditional': fmean(values) if values else None, **counts,
            'evaluated_fraction': counts['evaluated'] / counts['expected'],
            'metric_coverage_fraction': counts['metric_available'] / counts['expected'],
            'cohort_complete': all(row['cohort_complete'] for row in rows),
            'independent_geometry_count': 1, 'conditioning': rows[0]['conditioning'],
            'aggregation': 'equal_instance_weight_after_mean_over_solver_replicas',
            'cvar_scope': rows[0]['cvar_scope'],
        })
    return output


def _pairwise_tables(cases, reference):
    lookup = {(row['instance_sha256'], row['policy'], row['method'], row['metric']): row for row in cases}
    output = []
    for row in cases:
        if row['method'] == reference:
            continue
        ref = lookup[(row['instance_sha256'], row['policy'], reference, row['metric'])]
        complete = row['cohort_complete'] and ref['cohort_complete']
        delta = row['mean_conditional'] - ref['mean_conditional'] if complete else None
        ratio = 100. * delta / ref['mean_conditional'] if complete and ref['mean_conditional'] != 0 else None
        output.append({key: row[key] for key in ('instance_id', 'instance_sha256', 'geometry_group_sha256',
                                                 'n_customers', 'policy', 'method', 'metric')} | {
            'reference_method': reference, 'status': 'complete' if complete else 'incomplete_cohort',
            'method_mean': row['mean_conditional'] if complete else None,
            'reference_mean': ref['mean_conditional'] if complete else None,
            'delta_mean': delta, 'ratio_percent': ratio,
            'ratio_status': 'defined' if ratio is not None else ('zero_reference' if complete else 'incomplete_cohort'),
            'delta_percentage_points': 100 * delta if complete and row['metric'] == 'scenario_strict_feasibility' else None,
            'method_evaluated': row['evaluated'], 'reference_evaluated': ref['evaluated'],
            'expected_per_method': row['expected'],
            'direction': 'higher_is_better' if row['metric'] == 'scenario_strict_feasibility' else 'lower_is_better',
            'solver_seed_pairing': False,
            'aggregation': 'difference_of_complete_method_replica_means_within_case',
        })
    return output


def _scenario_tables(scenario_rows, index, registry, policies, seeds, methods, cases, reference):
    if not scenario_rows:
        return []
    by_run = defaultdict(dict)
    for row in scenario_rows:
        key = _row_key(row, registry, policies, methods, seeds)
        if _category(index.get(key)) != 'evaluated':
            raise ValueError('Escenarios sin una evaluación validated correspondiente.')
        scenario = _seed(row.get('scenario_seed'))
        if scenario in by_run[key]:
            raise ValueError('Escenario duplicado dentro de una réplica.')
        by_run[key][scenario] = row
    # Cada ejecución disponible debe conservar su evaluación íntegra. Los
    # escenarios omitidos no se convierten en una selección de casos fáciles.
    sets_by_case = defaultdict(list)
    for key, summary in index.items():
        if _category(summary) != 'evaluated':
            continue
        expected_count = _seed(summary.get('scenarios'))
        observed = by_run.get(key, {})
        if expected_count < 1 or len(observed) != expected_count:
            raise ValueError('Cohorte de escenarios incompleta frente al resumen validado.')
        sets_by_case[key[:2]].append(set(observed))
    for scenario_sets in sets_by_case.values():
        if any(scenarios != scenario_sets[0] for scenarios in scenario_sets[1:]):
            raise ValueError('Los métodos/réplicas no comparten los mismos escenarios CRN.')
    complete = {(row['instance_sha256'], row['policy'], row['method'], row['metric']): row['cohort_complete']
                for row in cases}
    output = []
    for sha, identity in registry.items():
        for policy in policies:
            scenario_sets = sets_by_case.get((sha, policy), [])
            if not scenario_sets:
                continue
            for method in methods:
                if method == reference:
                    continue
                for metric, source in SCENARIO_METRICS.items():
                    if not (complete[(sha, policy, method, metric)] and complete[(sha, policy, reference, metric)]):
                        continue
                    for scenario in sorted(scenario_sets[0]):
                        means = {}
                        for selected in (method, reference):
                            values = [_number(by_run[(sha, policy, selected, seed)][scenario].get(source),
                                              boolean=source == 'strict_feasible') for seed in seeds]
                            if any(value is None for value in values):
                                raise ValueError('Escenario validado con métrica ausente.')
                            if any(value < 0 or (source == 'strict_feasible' and value not in (0., 1.)) for value in values):
                                raise ValueError('Métrica de escenario fuera de su dominio.')
                            means[selected] = fmean(values)
                        delta = means[method] - means[reference]
                        output.append(_context(identity, policy, method) | {
                            'reference_method': reference, 'scenario_seed': str(scenario),
                            'metric': metric, 'scenario_metric': source,
                            'method_replica_mean': means[method], 'reference_replica_mean': means[reference],
                            'delta_mean': delta,
                            'delta_percentage_points': 100 * delta if source == 'strict_feasible' else None,
                            'replicas_per_method': len(seeds), 'pairing': 'scenario_seed_CRN_after_replica_means',
                            'solver_seed_pairing': False, 'independent_population_observation': False,
                        })
    return output


def _gap_tables(index, registry, policies, seeds, methods):
    bounds = defaultdict(list)
    for (sha, policy, method, seed), row in index.items():
        if method != 'gurobi' or row.get('solver_status') == 'license_pending':
            continue
        bound = _number(row.get('best_bound'))
        if bound is not None:
            bounds[sha].append((bound, seed, policy, row))
    output = []
    for sha, identity in registry.items():
        candidates = bounds.get(sha, [])
        source = max(candidates, key=lambda value: value[0]) if candidates else None
        lb = source[0] if source else None
        validated_optimum = False
        for bound, _, _, row in candidates:
            objective = _number(row.get('search_objective_minutes'))
            if (row.get('surrogate_optimal') is True and _category(row) == 'evaluated'
                    and row.get('structural_valid') is True and row.get('conservative_feasible') is True
                    and objective is not None and math.isclose(bound, objective,
                                                              rel_tol=_BOUND_REL_TOL, abs_tol=_BOUND_ABS_TOL)):
                validated_optimum = True
        for policy in policies:
            for method in methods:
                for seed in seeds:
                    row = index.get((sha, policy, method, seed))
                    candidate = _number(row.get('search_objective_minutes')) if row else None
                    eligible = bool(row and _category(row) == 'evaluated' and row.get('structural_valid') is True
                                    and row.get('conservative_feasible') is True and candidate is not None)
                    gap, clipped = None, False
                    if not eligible:
                        status = 'candidate_unavailable_or_not_conservative_feasible'
                    elif candidate < 0:
                        raise ValueError('Costo conservador negativo.')
                    elif lb is None:
                        status = 'no_gurobi_bound'
                    elif lb > candidate and not math.isclose(lb, candidate, rel_tol=_BOUND_REL_TOL, abs_tol=_BOUND_ABS_TOL):
                        status = 'bound_exceeds_candidate_inconsistent'
                    elif candidate == 0:
                        status = 'zero_candidate_ratio_undefined'
                    else:
                        clipped = lb > candidate
                        gap = max(0., (candidate - lb) / candidate * 100.)
                        status = 'available_upper_bound_on_relative_suboptimality'
                    output.append(_context(identity, policy, method) | {
                        'solver_seed': seed, 'candidate_cost_minutes': candidate if eligible else None,
                        'best_gurobi_lower_bound_minutes': lb, 'gap_upperbound_pct': gap, 'status': status,
                        'bound_aggregation': 'strongest_certified_bound_across_all_recorded_gurobi_attempts',
                        'bound_is_equal_single_attempt_runtime_comparison': False,
                        'bound_is_not_stochastic_mean_cost_reference': True,
                        'gap_denominator': 'candidate_conservative_objective_not_unknown_optimum',
                        'bound_source_solver_seed': source[1] if source else None,
                        'bound_source_policy': source[2] if source else None,
                        'gurobi_surrogate_optimum_validated': validated_optimum,
                        'candidate_surrogate_optimal_validated': bool(eligible and method == 'gurobi'
                            and row.get('surrogate_optimal') is True and lb is not None
                            and _number(row.get('best_bound')) is not None
                            and math.isclose(candidate, _number(row['best_bound']),
                                             rel_tol=_BOUND_REL_TOL, abs_tol=_BOUND_ABS_TOL)
                            and math.isclose(candidate, lb, rel_tol=_BOUND_REL_TOL, abs_tol=_BOUND_ABS_TOL)),
                        'numerical_equality_clipped_to_zero': clipped,
                        'scope': 'conservative_surrogate_same_instance_only_not_stochastic_optimality',
                        'numerical_equality_is_optimality_proof': False,
                    })
    return output


def _success_value(row):
    """Observed pipeline success; no-solution is zero, missing evaluation unknown."""
    if row is None or row.get('solver_status') == 'license_pending':
        return None
    if _category(row) == 'evaluated':
        return _number(row.get('scenario_strict_feasibility'))
    if row.get('evaluation_status') == 'no_solution' and row.get('structural_valid') is False:
        return 0.
    return None


def _joint_tables(index, registry, policies, seeds, methods):
    output = []
    for sha, identity in registry.items():
        for policy in policies:
            for method in methods:
                rows = [index.get((sha, policy, method, seed)) for seed in seeds]
                valid = [r for r in rows if _category(r) == 'evaluated']
                conservative = [r for r in valid if r.get('conservative_feasible') is True]
                values = [_success_value(r) for r in rows]
                observed = [v for v in values if v is not None]
                unresolved = len(rows) - len(observed)
                def mean(field, eligible):
                    data = [_number(r.get(field)) for r in eligible]
                    data = [v for v in data if v is not None]
                    return fmean(data) if data else None
                output.append(_context(identity, policy, method) | {
                    'expected_attempts': len(seeds),
                    'recorded_attempts': sum(r is not None for r in rows),
                    'evaluated_attempts': len(valid),
                    'conservative_feasible_attempts': len(conservative),
                    'conservative_feasible_per_scheduled_attempt': len(conservative) / len(seeds),
                    'nominal_feasible_attempts': sum(r.get('nominal_feasible') is True for r in valid),
                    'failed_attempts': sum(_category(r) == 'failed' for r in rows),
                    'missing_attempts': sum(r is None for r in rows),
                    'license_pending_attempts': sum(_category(r) == 'license_pending' for r in rows),
                    'scenario_success_known_attempts': len(observed),
                    'scenario_success_unresolved_attempts': unresolved,
                    'scenario_success_per_scheduled_attempt': sum(observed) / len(seeds) if not unresolved else None,
                    'scenario_success_lower_bound': sum(observed) / len(seeds),
                    'scenario_success_upper_bound': (sum(observed) + unresolved) / len(seeds),
                    'success_bounds_are_confidence_intervals': False,
                    'scenario_success_conditioning': 'all_scheduled_attempts_no_solution_zero_unresolved_bounded',
                    'mean_travel_conditional_evaluated': mean('mean_travel_minutes', valid),
                    'mean_travel_conditional_conservative_feasible': mean('mean_travel_minutes', conservative),
                    'cvar_travel_conditional_conservative_feasible': mean('cvar_travel_minutes', conservative),
                    'cost_denominator_evaluated': len(valid),
                    'cost_denominator_conservative_feasible': len(conservative),
                    'no_numeric_cost_assigned_to_failed_solution': True,
                })
    return output


def _contrasts(methods, config):
    pairs = [(m, config['reference_method']) for m in methods if m != config['reference_method']]
    controls = config.get('controlled_contrasts', [
        {'method': 'pomo', 'reference': 'pomo_untrained'},
        {'method': 'pomo', 'reference': 'random_masked'},
        {'method': 'hbg_facs', 'reference': 'hbg_no_pheromone'},
        {'method': 'hbg_facs', 'reference': 'hbg_untrained'},
        {'method': 'hbg_facs', 'reference': 'ortools_routing'},
        {'method': 'hbg_facs', 'reference': 'edd'},
    ])
    for contrast in controls:
        method, reference = contrast['method'], contrast['reference']
        if method not in METHODS or reference not in METHODS or method == reference:
            raise ValueError('Contraste controlado inválido.')
        if method in methods and reference in methods and (method, reference) not in pairs:
            pairs.append((method, reference))
    return pairs


def _paired_geometry_costs(index, registry, policies, seeds, methods, config):
    """Costs only on the same complete feasible geometries, all seeds retained.

    Selection here is explicit conditioning, never a claim over all geometries.
    Every expected geometry receives a row, including exclusions and their counts.
    """
    geometries = defaultdict(list)
    for sha, identity in registry.items():
        geometries[identity['geometry_group_sha256']].append(sha)
    output = []
    for method, reference in _contrasts(methods, config):
        for policy in policies:
            for metric in ('mean_travel_minutes', 'cvar_travel_minutes'):
                pair_rows = []
                for geometry, instances in geometries.items():
                    eligible, means, counts = {}, {}, {}
                    for selected in (method, reference):
                        per_instance = []
                        expected = len(instances) * len(seeds)
                        qualifying = 0
                        for sha in instances:
                            values = []
                            for seed in seeds:
                                r = index.get((sha, policy, selected, seed))
                                if (_category(r) == 'evaluated' and r.get('conservative_feasible') is True
                                        and r.get('structural_valid') is True):
                                    value = _number(r.get(metric))
                                    if value is not None:
                                        values.append(value)
                            qualifying += len(values)
                            if len(values) == len(seeds):
                                per_instance.append(fmean(values))
                        eligible[selected] = len(per_instance) == len(instances)
                        counts[selected] = qualifying
                        means[selected] = fmean(per_instance) if eligible[selected] else None
                    common = eligible[method] and eligible[reference]
                    pair_rows.append({
                        'geometry_group_sha256': geometry, 'policy': policy, 'metric': metric,
                        'method': method, 'reference_method': reference,
                        'status': 'common_complete_feasible' if common else 'excluded_incomplete_or_infeasible',
                        'expected_instances': len(instances), 'expected_attempts_per_method': expected,
                        'method_qualifying_attempts': counts[method], 'reference_qualifying_attempts': counts[reference],
                        'method_mean': means[method] if common else None,
                        'reference_mean': means[reference] if common else None,
                        'delta_mean': means[method] - means[reference] if common else None,
                        'conditional_on_both_methods_complete_conservative_feasibility': True,
                        'solver_seed_pairing': False, 'independent_geometry_count': 1,
                    })
                total = sum(r['status'] == 'common_complete_feasible' for r in pair_rows)
                for r in pair_rows:
                    r.update(common_feasible_geometry_count=total, expected_geometry_count=len(geometries))
                output.extend(pair_rows)
    return output


def _paired_geometry_success(index, registry, policies, seeds, methods, config):
    joint = _joint_tables(index, registry, policies, seeds, methods)
    lookup = {(r['instance_sha256'], r['policy'], r['method']): r for r in joint}
    geometries = defaultdict(list)
    for sha, identity in registry.items():
        geometries[identity['geometry_group_sha256']].append(sha)
    output = []
    for method, reference in _contrasts(methods, config):
        for policy in policies:
            for geometry, instances in geometries.items():
                values = {}
                for selected in (method, reference):
                    per_instance = [lookup[(sha, policy, selected)]['scenario_success_per_scheduled_attempt']
                                    for sha in instances]
                    values[selected] = fmean(per_instance) if all(v is not None for v in per_instance) else None
                complete = all(value is not None for value in values.values())
                output.append({'geometry_group_sha256': geometry, 'policy': policy,
                    'method': method, 'reference_method': reference,
                    'metric': 'scenario_success_per_scheduled_attempt',
                    'method_mean': values[method], 'reference_mean': values[reference],
                    'delta_percentage_points': 100 * (values[method] - values[reference]) if complete else None,
                    'status': 'complete_known_cohort' if complete else 'unresolved_attempts',
                    'failed_solutions_retained_as_zero_success': True,
                    'all_expected_solver_seeds_retained': True,
                    'expected_attempts_per_method': len(instances) * len(seeds),
                    'expected_instances': len(instances), 'independent_geometry_count': 1,
                    'aggregation': 'equal_instance_weight_after_all_expected_solver_seeds',
                })
    return output


def _risk_diagnostics(index):
    output = []
    for row in index.values():
        if _category(row) != 'evaluated':
            continue
        mean = _number(row.get('mean_travel_minutes'))
        cvar = _number(row.get('cvar_travel_minutes'))
        variance = _number(row.get('cost_variance_population'))
        std = _number(row.get('std_travel_minutes'))
        output.append({key: row.get(key) for key in (
            'instance_id', 'instance_sha256', 'geometry_group_sha256', 'n_customers',
            'policy', 'method', 'solver_seed', 'scenarios', 'alpha')} | {
            'mean_travel_minutes': mean, 'std_travel_minutes': std,
            'cost_variance_population': variance,
            'coefficient_of_variation': std / mean if std is not None and mean else None,
            'cvar_minus_mean_minutes': cvar - mean if cvar is not None and mean is not None else None,
            'cvar_excess_percent': 100 * (cvar - mean) / mean if cvar is not None and mean else None,
            'scope': 'conditional_scenario_variation_of_one_plan_not_geometries',
            'evidence_of_stochastic_optimality': False,
        })
    return output


def _recourse_differences(index):
    output = []
    for (sha, policy, method, seed), row in index.items():
        if policy != 'causal_edd':
            continue
        reference = index.get((sha, 'fixed_order', method, seed))
        available = _category(row) == _category(reference) == 'evaluated'
        for metric in ('scenario_strict_feasibility', 'mean_travel_minutes', 'cvar_travel_minutes',
                       'mean_lateness_minutes', 'mean_depot_overtime_minutes'):
            value = _number(row.get(metric)) if available else None
            base = _number(reference.get(metric)) if available else None
            output.append({key: row.get(key) for key in ('instance_id', 'instance_sha256',
                'geometry_group_sha256', 'n_customers', 'method', 'solver_seed')} | {
                'policy': policy, 'reference_policy': 'fixed_order', 'metric': metric,
                'status': 'paired_same_plan' if value is not None and base is not None else 'unavailable',
                'delta_policy_minus_fixed': value - base if value is not None and base is not None else None,
                'direction': 'higher_is_better' if metric == 'scenario_strict_feasibility' else 'lower_is_better',
                'same_plan_same_scenarios': True, 'independent_population_observation': False,
            })
    return output


def _budget_audit(index):
    attempts = {}
    for (sha, policy, method, seed), row in index.items():
        key = sha, method, seed
        fields = ('search_budget_seconds', 'search_total_seconds', 'solve_seconds', 'model_load_seconds',
                  'offline_cost_id', 'model_checkpoint_sha256', 'training_seconds', 'solver_status')
        values = {field: row.get(field) for field in fields}
        if key in attempts and any(attempts[key].get(k) != v for k, v in values.items()):
            raise ValueError('Costo/procedencia de un intento cambia entre políticas.')
        budget, total = _number(values['search_budget_seconds']), _number(values['search_total_seconds'])
        attempts[key] = {k: row.get(k) for k in ('instance_id', 'instance_sha256',
            'geometry_group_sha256', 'n_customers', 'method', 'solver_seed')} | values | {
            'budget_used_fraction': total / budget if total is not None and budget else None,
            'policy_rows_are_not_additional_searches': True,
            'same_seconds_do_not_imply_same_hardware_or_operations': True,
        }
    return list(attempts.values())


def _inference_readiness(registry, config):
    design = config.get('future_confirmatory_design', {})
    return [{
        'status': 'blocked_development_bank', 'current_role': config['role'],
        'independent_geometry_count': len({r['geometry_group_sha256'] for r in registry.values()}),
        'development_geometries_must_be_excluded_from_final_test': True,
        'minimum_independent_test_geometries': design.get('minimum_independent_test_geometries'),
        'sample_size_justification': design.get('sample_size_justification'),
        'preregistered_primary_estimand': design.get('primary_estimand'),
        'paired_resampling_unit': 'geometry_group_sha256',
        'requirements': 'sealed_disjoint_test; frozen_models_and_analysis_before_test; '
                        'independent_geometry_sample_size_justification; complete_all_seed_attempts; '
                        'predeclared_primary_estimand_and_contrasts; scenario_precision_plan',
        'bootstrap_or_tests_executed': False,
        'confirmatory_pipeline_implemented': False,
    }]


def build_tables(summary_rows, scenario_rows, registry, policies, seeds, config, *, contrast_audit=None):
    """Tablas largas descriptivas con cobertura explícita del diseño esperado.

    Ausentes son null, nunca cero. Tiempo se describe sobre intentos observados,
    incluidos fallos; FR se describe sobre evaluaciones válidas y costos sobre
    planes conservadoramente factibles. joint_performance conserva ambos denominadores.
    Las diferencias de métodos requieren todas las réplicas válidas de ambos.
    No se producen agregados globales que confundan tamaños con replicaciones.
    """
    methods, metrics = _config(config)
    registry = _identity(registry)
    policies, seeds = list(policies), [_seed(seed) for seed in seeds]
    if (not policies or any(not isinstance(policy, str) or not policy for policy in policies)
            or len(set(policies)) != len(policies) or not seeds or len(set(seeds)) != len(seeds)):
        raise ValueError('Políticas/semillas esperadas vacías o duplicadas.')
    index = {}
    for row in summary_rows:
        key = _row_key(row, registry, policies, methods, seeds)
        if key in index:
            raise ValueError('Réplica duplicada: no promediar dos reportes del mismo intento.')
        index[key] = dict(row)
    cases = _case_tables(index, registry, policies, seeds, methods, metrics)
    reference = config['reference_method']
    tables = {
        'case_summary': cases,
        'geometry_summary': _geometry_tables(cases),
        'pairwise': _pairwise_tables(cases, reference),
        'scenario_differences': _scenario_tables(scenario_rows, index, registry, policies, seeds, methods, cases, reference),
        'gaps': _gap_tables(index, registry, policies, seeds, methods),
        'joint_performance': _joint_tables(index, registry, policies, seeds, methods),
        'paired_geometry_costs': _paired_geometry_costs(index, registry, policies, seeds, methods, config),
        'paired_geometry_success': _paired_geometry_success(index, registry, policies, seeds, methods, config),
        'risk_diagnostics': _risk_diagnostics(index),
        'recourse_differences': _recourse_differences(index),
        'budget_audit': _budget_audit(index),
        'inference_readiness': _inference_readiness(registry, config),
        'controlled_contrast_audit': list(contrast_audit or []),
        'statistical_scope': {
            'schema_version': SCHEMA, 'role': 'development_only', 'analysis_mode': 'descriptive_only',
            'ready_for_confirmatory': False, 'no_pvalues': True, 'no_population_bootstrap': True,
            'no_population_confidence_intervals': True, 'no_global_ranking': True,
            'independent_geometry_count': len({row['geometry_group_sha256'] for row in registry.values()}),
            'instance_count': len(registry), 'expected_solver_replicas_per_case_method': len(seeds),
            'expected_methods': methods, 'reference_method': reference, 'metrics': metrics,
            'scenario_comparison_status': 'available' if scenario_rows else 'not_provided',
            'unit_of_independence': 'geometry_group_sha256',
            'reference_selection_note': config.get('reference_selection_note', 'historical_descriptive_reference'),
            'solver_seeds_are_equivalent_rng_pairs': False,
            'scenarios_or_seeds_increase_independent_geometry_count': False,
            'conditional_means_are_not_complete_cohort_rankings': True,
            'failed_attempts_count_in_end_to_end_success': True,
            'missing_or_unevaluated_success_is_bounded_not_imputed': True,
            'no_artificial_cost_penalty': True,
            'controlled_effects_require_verified_matched_design': True,
            'controls': [name for name in methods if name in CONTROLS],
            'cvar_aggregation': 'mean_of_per_plan_cvar_never_pooled_plan_scenarios',
            'gap_scope': 'conservative_surrogate_bound_only_not_stochastic_optimality',
            'bound_equality_relative_tolerance': _BOUND_REL_TOL,
            'bound_equality_absolute_tolerance_minutes': _BOUND_ABS_TOL,
            'numerical_bound_equality_alone_proves_optimality': False,
            'exact_equality_note': 'Exact equality of a certified lower bound and a feasible objective '
                                   'would prove surrogate optimality; a floating tolerance comparison is not that certificate.',
        },
    }

    for table_name in ('paired_geometry_costs', 'paired_geometry_success'):
        for row in tables[table_name]:
            if (row['method'], row['reference_method']) not in CONTROLLED_PAIRS:
                row['contrast_design_status'] = 'comparison_not_single_factor_ablation'
                continue
            matching = [r for r in tables['controlled_contrast_audit']
                        if r['geometry_group_sha256'] == row['geometry_group_sha256']
                        and r['method'] == row['method'] and r['reference_method'] == row['reference_method']]
            verified = (len(matching) == row['expected_instances'] and
                        len({r['instance_sha256'] for r in matching}) == row['expected_instances'] and
                        all(r['status'] == 'verified' for r in matching))
            row['contrast_design_status'] = 'verified' if verified else 'unverified'
            if not verified:
                row['data_cohort_status'] = row['status']
                row['status'] = 'unverified_control_design'
                for key in ('method_mean', 'reference_mean', 'delta_mean', 'delta_percentage_points'):
                    if key in row:
                        row[key] = None
        if table_name == 'paired_geometry_costs':
            counts = defaultdict(int)
            for row in tables[table_name]:
                if row['status'] == 'common_complete_feasible':
                    counts[(row['method'], row['reference_method'], row['policy'], row['metric'])] += 1
            for row in tables[table_name]:
                row['common_feasible_geometry_count'] = counts[(row['method'], row['reference_method'], row['policy'], row['metric'])]
    return tables


def save_plots(tables, directory):
    """PNG descriptivos por geometría/política, sin intervalos ni ranking global."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    rows = tables['geometry_summary']
    methods = tables['statistical_scope']['expected_methods']
    lookup = {(r['geometry_group_sha256'], r['policy'], r['method'], r['metric']): r for r in rows}
    groups = sorted({(r['geometry_group_sha256'], r['policy']) for r in rows})
    files = []
    for geometry, policy in groups:
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        axes = axes.flatten()
        cost_colors = ['#3173a3', '#67a6c9']
        for i, (metric, label) in enumerate((('mean_travel_minutes', 'Media'), ('cvar_travel_minutes', 'CVaR por plan'))):
            for j, method in enumerate(methods):
                row = lookup[(geometry, policy, method, metric)]
                if row['mean_conditional'] is not None:
                    axes[0].bar(j + (i - .5) * .34, row['mean_conditional'], width=.32,
                                color=cost_colors[i], hatch=None if row['cohort_complete'] else '//',
                                edgecolor='#333333', linewidth=.5)
        for j, method in enumerate(methods):
            cost_rows = [lookup[(geometry, policy, method, metric)]
                         for metric in ('mean_travel_minutes', 'cvar_travel_minutes')]
            if all(row['mean_conditional'] is None for row in cost_rows):
                axes[0].text(j, .02, 'sin datos', ha='center', va='bottom', rotation=90,
                             fontsize=8, transform=axes[0].get_xaxis_transform())
            row = lookup[(geometry, policy, method, 'scenario_strict_feasibility')]
            if row['mean_conditional'] is None:
                axes[1].text(j, .02, 'sin datos', ha='center', va='bottom', rotation=90, fontsize=8)
            else:
                axes[1].bar(j, row['mean_conditional'], color='#3a8f79', edgecolor='#333333', linewidth=.5,
                            hatch=None if row['cohort_complete'] else '//')
        categories = [('evaluated', '#3a8f79', 'Evaluada'), ('failed', '#d57745', 'Fallo'),
                      ('license_pending', '#d3ac49', 'Licencia'), ('missing', '#c6c8cc', 'Faltante')]
        bottom = [0.] * len(methods)
        for key, color, label in categories:
            values = [lookup[(geometry, policy, method, 'scenario_strict_feasibility')][key] /
                      lookup[(geometry, policy, method, 'scenario_strict_feasibility')]['expected'] for method in methods]
            axes[2].bar(range(len(methods)), values, bottom=bottom, label=label, color=color)
            bottom = [old + value for old, value in zip(bottom, values)]
        for j, method in enumerate(methods):
            row = lookup[(geometry, policy, method, 'scenario_strict_feasibility')]
            axes[2].text(j, 1.02, f"{row['evaluated']}/{row['expected']}", ha='center', fontsize=8)
        axes[0].set(title='Viaje: condicionado a planes conservadores factibles', ylabel='Minutos')
        axes[0].legend(handles=[Patch(facecolor=cost_colors[0], label='Media'),
                                Patch(facecolor=cost_colors[1], label='CVaR por plan'),
                                Patch(facecolor='white', edgecolor='#333333', hatch='//', label='Cohorte incompleta')], fontsize=8)
        axes[1].set(title='Factibilidad condicional', ylabel='Fracción de escenarios', ylim=(0, 1.08))
        axes[2].set(title='Cobertura de réplicas esperadas', ylabel='Fracción de réplicas', ylim=(0, 1.12))
        axes[2].legend(fontsize=8, loc='lower left')
        joint = [r for r in tables['joint_performance'] if r['geometry_group_sha256'] == geometry and r['policy'] == policy]
        for j, method in enumerate(methods):
            selected = [r for r in joint if r['method'] == method]
            lo = fmean(r['scenario_success_lower_bound'] for r in selected)
            hi = fmean(r['scenario_success_upper_bound'] for r in selected)
            axes[3].bar(j, lo, color='#3173a3', edgecolor='#333333', linewidth=.5)
            axes[3].bar(j, hi - lo, bottom=lo, color='white', hatch='...', edgecolor='#777777', linewidth=.5)
        axes[3].set(title='Éxito incluyendo intentos sin solución', ylabel='Fracción de escenarios × intentos previstos', ylim=(0, 1.08))
        axes[3].legend(handles=[Patch(facecolor='#3173a3', label='Éxito observado / previstos'),
                                Patch(facecolor='white', edgecolor='#777777', hatch='...', label='Faltantes o evaluación desconocida')], fontsize=8)
        for axis in axes:
            axis.set_xticks(range(len(methods)), methods, rotation=45, ha='right', fontsize=8)
            axis.grid(axis='y', alpha=.18)
            axis.set_axisbelow(True)
        sizes = sorted({r['n_customers'] for r in rows if r['geometry_group_sha256'] == geometry})
        fig.suptitle(f'Geometría {geometry[:12]} · n={",".join(map(str, sizes))} · {policy} · N=1 geometría; desarrollo descriptivo', fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, .94))
        token = hashlib.sha256(f'{geometry}|{policy}'.encode()).hexdigest()[:12]
        path = directory / f'geometry_{token}_{re.sub(r"[^A-Za-z0-9_-]", "_", policy)}.png'
        for extension in ('png', 'pdf', 'svg'):
            exported = path.with_suffix('.' + extension)
            fig.savefig(exported, dpi=300, bbox_inches='tight')
            files.append(exported.name)
        plt.close(fig)
    # Paired cost plots show each geometry as one point and retain excluded counts.
    paired = tables['paired_geometry_costs']
    for policy in sorted({r['policy'] for r in paired}):
        selected = [r for r in paired if r['policy'] == policy and r['metric'] == 'mean_travel_minutes']
        contrasts = list(dict.fromkeys((r['method'], r['reference_method']) for r in selected))
        fig, panels = plt.subplots(1, 2, figsize=(18, max(5, .55 * len(contrasts))), sharey=True)
        ax = panels[0]
        for j, (method, reference) in enumerate(contrasts):
            rows = [r for r in selected if r['method'] == method and r['reference_method'] == reference]
            values = [r['delta_mean'] for r in rows if r['delta_mean'] is not None]
            for i, value in enumerate(values):
                offset = (i - (len(values) - 1) / 2) * .07
                ax.scatter(value, j + offset, s=38, color='#3173a3', alpha=.85)
            ax.text(1.01, j, f"{len(values)}/{len(rows)} geom.", transform=ax.get_yaxis_transform(), va='center', fontsize=9)
        ax.axvline(0, color='#666666', linewidth=1)
        # Empty contrasts still occupy rows; autoscale from points alone hides them.
        ax.set_ylim(-.6, len(contrasts) - .4)
        ax.set_yticks(range(len(contrasts)), [f'{m} − {r}' for m, r in contrasts])
        ax.set(xlabel='Diferencia de viaje medio (minutos); valores negativos favorecen al primer método',
               title=f'{policy}: costo condicionado a geometrías completas y factibles en ambos métodos')
        ax.grid(axis='x', alpha=.2)
        for j, (method, reference) in enumerate(contrasts):
            rows = [r for r in tables['paired_geometry_success'] if r['policy'] == policy
                    and r['method'] == method and r['reference_method'] == reference]
            values = [r['delta_percentage_points'] for r in rows if r['delta_percentage_points'] is not None]
            for i, value in enumerate(values):
                panels[1].scatter(value, j + (i - (len(values) - 1) / 2) * .07, s=38, color='#3a8f79', alpha=.85)
            panels[1].text(1.01, j, f'{len(values)}/{len(rows)} geom.', transform=panels[1].get_yaxis_transform(), va='center', fontsize=9)
        panels[1].axvline(0, color='#666666', linewidth=1)
        panels[1].set(xlabel='Diferencia de éxito (pp); positivos favorecen al primer método',
                      title='Éxito con todas las semillas; sin solución cuenta como cero')
        panels[1].grid(axis='x', alpha=.2)
        fig.tight_layout()
        for extension in ('png', 'pdf', 'svg'):
            path = directory / f'paired_cost_{re.sub(r"[^A-Za-z0-9_-]", "_", policy)}.{extension}'
            fig.savefig(path, dpi=300, bbox_inches='tight')
            files.append(path.name)
        plt.close(fig)
    return files


Writing svrplab/comparison_stats.py


In [8]:
%%writefile svrplab/comparison.py
"""06: lectura verificable de resultados y comparación descriptiva, sólo en Colab.

No resuelve, entrena, reevalúa escenarios ni accede a certificados. Reconciliar
los CSV es aritmética sobre mediciones ya exportadas por el evaluador sellado.
"""
from __future__ import annotations

import csv
import json
import math
import shutil
from pathlib import Path

from . import pilot as p
from . import experiments as e
from . import comparison_stats as stats

SCHEMA = 'comparison-06-v2'
LEARNED = {'am_supervised': 'supervised_training', 'pomo': 'rl_training',
           'hbg_facs': 'hbg_training', 'hbg_no_pheromone': 'hbg_training'}
FLOAT_FIELDS = ('travel_minutes', 'base_minutes', 'net_execution_delta_minutes',
                'waiting_minutes', 'lateness_minutes', 'depot_overtime_minutes',
                'customer_violation_fraction')
INT_FIELDS = ('late_customers', 'used_vehicles', 'recourse_reorders')
BOOL_FIELDS = ('strict_feasible', 'all_customers_served')


def _require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Análisis y pruebas exclusivamente en Google Colab.') from exc


def validate_config(config):
    # Una sola definición del catálogo/métricas evita aceptar opciones que las
    # tablas rechazarían después de leer todas las exportaciones.
    stats._config(config)
    if (config.get('schema_version') != SCHEMA or config.get('role') != 'development_only'
            or config.get('analysis_mode') != 'descriptive_only'):
        raise ValueError('06 sólo implementa comparación descriptiva del piloto de desarrollo.')
    methods = config.get('expected_methods', [])
    allowed = set(stats.METHODS)
    if (not methods or len(set(methods)) != len(methods) or not set(methods) <= allowed
            or config.get('reference_method') not in methods):
        raise ValueError('Métodos o referencia inválidos.')
    metrics = config.get('metrics', [])
    if (not metrics or len(set(metrics)) != len(metrics)
            or not set(metrics) <= set(e.SUMMARY_METRICS + ['search_total_seconds'])):
        raise ValueError('Métricas inválidas.')
    e.validate_config(config['method_contract'])
    ref = config['accepted_bank']
    if (ref['role'] != 'development_only'
            or ref['acceptance_status'] != 'ready_for_method_development'
            or len(ref['manifest_sha256']) != 64):
        raise ValueError('Referencia de banco no aceptada para desarrollo.')


def _safe_relative(root, relative):
    relative = Path(relative)
    root = Path(root).resolve()
    if relative.is_absolute() or '..' in relative.parts or not relative.parts:
        raise ValueError('Ruta relativa inválida en inventario.')
    path = root / relative
    if not path.resolve().is_relative_to(root):
        raise ValueError('Ruta fuera del directorio sellado.')
    return path


def _verify(directory):
    directory = Path(directory)
    manifest = e.read_json(directory / 'CHECKSUMS.json')
    if not isinstance(manifest, dict) or not manifest:
        raise ValueError('Manifiesto vacío o inválido.')
    for relative in manifest:
        _safe_relative(directory, relative)
    return p.verify_bundle(directory)


def _bank_context(bank, config):
    validate_config(config)
    bank = Path(bank)
    ref = config['accepted_bank']
    digest = _verify(bank)
    if bank.name != ref['bank_id'] or digest != ref['manifest_sha256']:
        raise ValueError('Banco distinto de la referencia aceptada en Colab.')
    bc = e.read_json(bank / 'config.json')
    acceptance = e.read_json(bank / 'acceptance.json')
    if (bc['protocol_version'] != ref['protocol_version']
            or acceptance.get('status') != ref['acceptance_status']):
        raise ValueError('Acta o versión del banco incompatibles.')
    evaluator = p.file_hash(bank / 'code/svrplab_pilot.py')
    if evaluator != p.file_hash(p.__file__):
        raise ValueError('06 debe importar el evaluador guardado en el banco.')
    # Sólo deserializar instancias: load_bank ejecutaría auditoría de certificados.
    instances = [p.freeze_instance(e.read_json(path))
                 for path in sorted((bank / 'instances').glob('*.json'))]
    if sorted(x.n for x in instances) != sorted(bc['sizes']):
        raise ValueError('Registro de tamaños incompleto.')
    registry = [{'instance_id': x.instance_id, 'instance_sha256': x.sha,
                 'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                 'n_customers': x.n} for x in instances]
    if (acceptance.get('instance_hashes') != {str(x.n): x.sha for x in instances}
            or acceptance.get('geometry_groups') != {str(x.n): x.provenance['geometry_group_sha256'] for x in instances}
            or acceptance.get('ready_for_confirmatory_comparison') is not False):
        raise ValueError('Registro de instancias distinto del acta sellada.')
    scenarios = {}
    for x in instances:
        split = e.read_json(bank / f'scenarios/n{x.n:04d}.json')
        e.validate_scenarios(split, bc)
        scenarios[x.sha] = split['evaluation']
    protocol = e.comparison_protocol(digest, evaluator, bc, config['method_contract'])
    return {'bank_id': bank.name, 'bank_hash': digest, 'bank_config': bc,
            'instances': instances, 'registry': registry, 'scenarios': scenarios,
            'protocol': protocol, 'comparison_id': p.digest(protocol),
            'method_contract': config['method_contract']}


def _validate_method_settings(plan, expected):
    for method in plan['methods']:
        if method not in expected or plan['config']['methods'].get(method) != expected[method]:
            raise ValueError(f'Protocolo distinto: parámetros no preespecificados para {method}.')


def _controlled_contrast_audit(rows, diagnostics, registry, seeds, method_configs, source_by_method):
    """Ablation claims require identical inference and verified model provenance."""
    index = {(r['instance_sha256'], r['method'], r['solver_seed']): r for r in rows}
    meta = {(r['instance_sha256'], r['method'], r['solver_seed']):
            json.loads(r['solver_metadata_json']) for r in diagnostics}
    output = []
    for method, reference in stats.CONTROLLED_PAIRS:
        family = 'pomo' if method == 'pomo' else 'hbg'
        required_sources = ('attention.py', 'pomo.py', 'reinforcement.py') if family == 'pomo' else (
                            'attention.py', 'hbg.py', 'hybrid.py', 'baselines.py')
        for identity in registry:
            reasons = set()
            settings = [dict(method_configs.get(m, {})) for m in (method, reference)]
            if reference == 'hbg_no_pheromone':
                if settings[0].get('alpha') != 1. or settings[1].get('alpha') != 0.:
                    reasons.add('pheromone_alpha_not_predeclared_1_vs_0')
                for value in settings:
                    value.pop('alpha', None)
            if not all(settings) or settings[0] != settings[1]:
                reasons.add('inference_parameters_differ_or_missing')
            environments = [source_by_method.get(m, {}).get('__environment_sha256') for m in (method, reference)]
            if not environments[0] or environments[0] != environments[1]:
                reasons.add('execution_environment_differs_or_missing')
            for source in required_sources:
                versions = [source_by_method.get(m, {}).get(source) for m in (method, reference)]
                if not versions[0] or versions[0] != versions[1]:
                    reasons.add('implementation_sources_differ_or_missing')
            for seed in seeds:
                keys = [(identity['instance_sha256'], m, seed) for m in (method, reference)]
                records = [index.get(k) for k in keys]
                metadata = [meta.get(k) for k in keys]
                if not all(records) or not all(metadata):
                    reasons.add('attempt_or_metadata_missing')
                    continue
                if reference == 'hbg_no_pheromone':
                    for field in ('training_id', 'model_checkpoint_sha256', 'fold_id', 'training_seed',
                                  'offline_manifest_sha256'):
                        if records[0].get(field) is None or records[0].get(field) != records[1].get(field):
                            reasons.add('trained_checkpoint_or_training_identity_differs')
                # Shape, masking, and initialization are recorded by solver wrappers
                # even if no route is produced. Missing proof is not guessed from names.
                for field in ('model_config', 'mask_version', 'inference_device'):
                    values = [m.get(field) for m in metadata]
                    if values[0] is None or values[0] != values[1]:
                        reasons.add(f'{field}_differs_or_missing')
                if reference != 'hbg_no_pheromone':
                    fold = records[0].get('fold_id')
                    expected_seed = p.seed_for('reinforcement-04' if family == 'pomo' else 'hybrid-05', fold, seed) if fold else None
                    values = [m.get('initialization_seed') for m in metadata]
                    if expected_seed is None or values != [expected_seed, expected_seed]:
                        reasons.add('initialization_seed_differs_or_unverified')
                    if records[1].get('training_seconds', 0.) != 0.:
                        reasons.add('untrained_control_has_training_cost')
            output.append({**identity, 'method': method, 'reference_method': reference,
                'status': 'verified' if not reasons else 'unverified',
                'reason': ';'.join(sorted(reasons)) if reasons else 'same_inference_sources_and_required_model_provenance',
                'expected_solver_seeds': len(seeds),
                'attribution_scope': 'pheromone_with_same_weights' if reference == 'hbg_no_pheromone' else
                                     'learning_or_policy_control_with_same_inference_and_initialization',
            })
    return output


def _inspect_run(directory, bank_context):
    directory = Path(directory)
    meta_hash = _verify(directory / 'meta')
    report_hash = _verify(directory / 'report')
    plan = e.read_json(directory / 'meta/plan.json')
    if (plan['run_id'] != directory.name or plan['bank_id'] != bank_context['bank_id']
            or plan['role'] != 'development_only'):
        raise ValueError('Identidad del run inválida.')
    e.validate_config(plan['config'])
    expected = e.comparison_protocol(bank_context['bank_hash'],
        bank_context['protocol']['evaluator_sha256'], bank_context['bank_config'], plan['config'])
    if (plan['protocol'] != expected or plan['protocol'] != bank_context['protocol']
            or plan['comparison_id'] != p.digest(expected)):
        raise ValueError('Protocolo distinto: banco/evaluador/runner/objetivo/presupuesto/CRN.')
    _validate_method_settings(plan, bank_context['method_contract']['methods'])
    methods = plan['methods']
    if (not methods or len(set(methods)) != len(methods)
            or not set(methods) <= set(plan['config']['methods'])
            or set(plan['method_availability']) != set(methods)
            or not set(plan['method_availability'].values()) <= {'enabled', 'license_pending'}):
        raise ValueError('Inventario de métodos inválido.')
    signature = p.digest({key: plan[key] for key in
        ['protocol', 'config', 'methods', 'source_hashes', 'method_availability', 'method_provenance']})
    if plan['signature'] != signature:
        raise ValueError('Firma de ejecución inconsistente.')
    for name, digest in plan['source_hashes'].items():
        if p.file_hash(_safe_relative(directory / 'meta/code', name)) != digest:
            raise ValueError('Fuente guardada distinta del manifiesto.')
    for name, key in [('experiments.py', 'runner_sha256'), ('search.py', 'search_contract_sha256'),
                      ('pilot.py', 'evaluator_sha256')]:
        if plan['source_hashes'].get(name) != plan['protocol'][key]:
            raise ValueError('Fuentes del run no corresponden al protocolo.')
    validation = e.read_json(directory / 'report/validation.json')
    if (validation.get('status') != 'completed' or validation.get('bank_unchanged') is not True
            or validation.get('ready_for_confirmatory_comparison') is not False):
        raise ValueError('Ejecución sin validación completa de desarrollo.')
    bundles = e.read_json(directory / 'report/bundles.json')
    if bundles.get('meta') != meta_hash:
        raise ValueError('Índice de bundles no corresponde al meta.')
    return {'plan': plan, 'bundles': bundles, 'validation': validation,
            'report_sha256': report_hash, 'meta_sha256': meta_hash}


def discover_runs(bank, method_runs_root, config):
    """Inventario ligero: la carga final verifica además todos los bundles/CSV."""
    _require_colab()
    bc = _bank_context(bank, config)
    inventory = []
    root = Path(method_runs_root)
    for directory in sorted(root.iterdir()) if root.exists() else []:
        if not directory.is_dir():
            continue
        item = {'run_id': directory.name, 'methods': [], 'comparison_id': None,
                'status': 'incomplete', 'reason': 'Falta meta o reporte sellado.'}
        try:
            if not all((directory / part / 'CHECKSUMS.json').is_file() for part in ['meta', 'report']):
                inventory.append(item)
                continue
            plan = e.read_json(directory / 'meta/plan.json')
            item.update(methods=plan.get('methods', []), comparison_id=plan.get('comparison_id'))
            inspected = _inspect_run(directory, bc)
            item.update(status='compatible', reason='Meta/reporte verificados; bundles se verifican al cargar.',
                        report_sha256=inspected['report_sha256'])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            item.update(status='incompatible' if 'Protocolo distinto' in str(exc) else 'invalid',
                        reason=str(exc))
        inventory.append(item)
    return inventory


def select_runs(inventory, config, selection=None):
    """No selecciona por fecha, rendimiento o éxito; exige resolver duplicados."""
    validate_config(config)
    selection = {} if selection is None else selection
    if not isinstance(selection, dict) or not set(selection) <= set(config['expected_methods']):
        raise ValueError('RUN_SELECTION debe mapear métodos esperados a run_id.')
    result, ids = {}, set()
    for method in config['expected_methods']:
        options = [r for r in inventory if r['status'] == 'compatible' and method in r['methods']]
        if method in selection:
            options = [r for r in options if r['run_id'] == selection[method]]
            if len(options) != 1:
                raise ValueError(f'Run seleccionado no compatible para {method}.')
        elif len(options) > 1:
            raise ValueError(f'Varios runs para {method}; fije RUN_SELECTION explícitamente: '
                             + ', '.join(r['run_id'] for r in options))
        if options:
            result[method] = options[0]['run_id']
            ids.add(options[0]['comparison_id'])
    if not result:
        raise ValueError('No hay runs compatibles. Ejecute 01–05 en Colab y revise el inventario.')
    if len(ids) != 1 or None in ids:
        raise ValueError('No se pueden mezclar comparison_id diferentes.')
    return result


def _finite(value, key, nonnegative=False):
    if isinstance(value, bool):
        raise ValueError(f'Número booleano en {key}.')
    try:
        number = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'Número inválido en {key}.') from exc
    if not math.isfinite(number) or (nonnegative and number < 0):
        raise ValueError(f'Número no finito/negativo en {key}.')
    return number


def _read_scenarios(path, context, expected_seeds):
    records, observed = [], []
    required = {'scenario_seed', *FLOAT_FIELDS, *INT_FIELDS, *BOOL_FIELDS, *context}
    with Path(path).open(newline='', encoding='utf-8') as stream:
        reader = csv.DictReader(stream)
        if (not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames)
                or not required <= set(reader.fieldnames)):
            raise ValueError('Columnas de escenarios incompletas/duplicadas.')
        for row in reader:
            if None in row or any(value is None for value in row.values()):
                raise ValueError('Fila CSV malformada.')
            for key, value in context.items():
                if row[key] != ('' if value is None else str(value)):
                    raise ValueError(f'Escenario con identidad distinta: {key}.')
            seed = row['scenario_seed']
            if not seed.isascii() or not seed.isdecimal() or str(int(seed)) != seed:
                raise ValueError('Semilla no es decimal entero exacto.')
            observed.append(int(seed))
            for key in FLOAT_FIELDS:
                row[key] = _finite(row[key], key, key != 'net_execution_delta_minutes')
            for key in INT_FIELDS:
                if not row[key].isascii() or not row[key].isdecimal():
                    raise ValueError(f'Conteo inválido en {key}.')
                row[key] = int(row[key])
            for key in BOOL_FIELDS:
                if row[key] not in ('True', 'False'):
                    raise ValueError(f'Booleano inválido en {key}.')
                row[key] = row[key] == 'True'
            if row['customer_violation_fraction'] > 1 or not row['all_customers_served']:
                raise ValueError('Cobertura o fracción de violación inválida.')
            if (not math.isclose(row['net_execution_delta_minutes'],
                                  row['travel_minutes'] - row['base_minutes'], rel_tol=1e-9, abs_tol=1e-7)
                    or row['strict_feasible'] != (row['late_customers'] == 0 and row['depot_overtime_minutes'] <= 1e-9)):
                raise ValueError('Métricas de escenario internamente inconsistentes.')
            if 'n_customers' in context and (row['late_customers'] > context['n_customers']
                    or not math.isclose(row['customer_violation_fraction'],
                                        row['late_customers'] / context['n_customers'], abs_tol=1e-12)):
                raise ValueError('Fracción de clientes tardíos inconsistente.')
            # Restaurar tipos de identidad (solver_seed, n_customers, etc.).
            row.update(context)
            records.append(row)
    if observed != list(expected_seeds) or len(set(observed)) != len(observed):
        raise ValueError('Escenarios faltantes, duplicados, ajenos o reordenados.')
    return records


def _check_aggregates(rows, summary, alpha):
    if not rows or not 0 <= alpha < 1:
        raise ValueError('Muestra/alpha inválidos.')
    costs = [r['travel_minutes'] for r in rows]
    mean = math.fsum(costs) / len(costs)
    variance = math.fsum((v - mean) ** 2 for v in costs) / len(costs)
    descending = sorted(costs, reverse=True)
    mass = len(costs) * (1 - alpha)
    whole = min(len(costs), math.floor(mass))
    tail = math.fsum(descending[:whole])
    if whole < len(costs):
        tail += (mass - whole) * descending[whole]
    calculated = {'scenarios': len(rows), 'alpha': alpha, 'mean_travel_minutes': mean,
                  'cvar_travel_minutes': tail / mass, 'cost_variance_population': variance,
                  'std_travel_minutes': math.sqrt(variance),
                  'scenario_strict_feasibility': sum(r['strict_feasible'] for r in rows) / len(rows),
                  'used_vehicles': rows[0]['used_vehicles']}
    for raw in ['customer_violation_fraction', 'lateness_minutes', 'depot_overtime_minutes',
                'waiting_minutes', 'base_minutes', 'net_execution_delta_minutes',
                'late_customers', 'recourse_reorders']:
        calculated['mean_' + raw] = math.fsum(r[raw] for r in rows) / len(rows)
    if len({r['used_vehicles'] for r in rows}) != 1:
        raise ValueError('Cambió el número de vehículos entre escenarios.')
    for key, value in calculated.items():
        if not math.isclose(_finite(summary.get(key), key), value, rel_tol=1e-9, abs_tol=1e-7):
            raise ValueError(f'Resumen no reconcilia escenarios: {key}.')
    return calculated


def _load_run(directory, bc):
    inspected = _inspect_run(directory, bc)
    plan, bundles = inspected['plan'], inspected['bundles']
    for relative, digest in bundles.items():
        if _verify(_safe_relative(directory, relative)) != digest:
            raise ValueError(f'Bundle modificado: {relative}.')
    rows = e.read_json(Path(directory) / 'report/summary.json')
    if not isinstance(rows, list):
        raise ValueError('Resumen debe ser una lista de intentos.')
    indexed = {}
    for row in rows:
        key = (row['instance_sha256'], row['method'], row['solver_seed'], row['policy'])
        if key in indexed:
            raise ValueError('Intento/política duplicado en reporte.')
        indexed[key] = row
    expected_keys, expected_bundles, scenario_rows = set(), {'meta'}, []
    for x in bc['instances']:
        for method in plan['methods']:
            for seed in plan['config']['seeds']:
                job = f'{method}/n{x.n:04d}/seed_{seed}'
                expected_bundles.add(f'{job}/solution')
                solution_dir = _safe_relative(directory, f'{job}/solution')
                solution = e.read_json(solution_dir / 'solution.json')
                learned = plan['method_provenance'].get(method, {}).get(f'{x.sha}:{seed}', {})
                if method in LEARNED:
                    e.validate_learned_provenance(learned, x, seed)
                context = {'schema_version': e.SCHEMA, 'comparison_id': plan['comparison_id'],
                    'run_id': plan['run_id'], 'bank_id': bc['bank_id'],
                    'bank_manifest_sha256': bc['bank_hash'],
                    'evaluator_sha256': plan['protocol']['evaluator_sha256'], 'method': method,
                    'solver_seed': seed, 'instance_id': x.instance_id, 'instance_sha256': x.sha,
                    'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                    'n_customers': x.n, 'objective_scope': plan['protocol']['objective'],
                    'search_budget_seconds': plan['protocol']['search_seconds'],
                    'scenario_set_sha256': p.digest(bc['scenarios'][x.sha]),
                    **e.provenance_columns(learned)}
                if e.read_json(solution_dir / 'identity.json') != context:
                    raise ValueError('Identidad de solución inconsistente.')
                meta, nominal = solution['metadata'], solution['nominal']
                if plan['method_availability'][method] == 'license_pending' and (
                        solution['status'] != 'license_pending' or solution['routes'] is not None):
                    raise ValueError('Disponibilidad de licencia inconsistente.')
                base = {**context, 'solver_status': solution['status'],
                    'structural_valid': solution['routes'] is not None,
                    'nominal_feasible': nominal['strict_feasible'] if nominal else None,
                    'conservative_feasible': meta.get('conservative_feasible'),
                    'search_violation_minutes': meta.get('search_violation_minutes'),
                    'search_objective_minutes': meta.get('search_objective_minutes'),
                    'solve_seconds': solution['solve_seconds'],
                    'search_total_seconds': solution['search_total_seconds'],
                    'best_bound': meta.get('best_bound'), 'relative_gap': meta.get('relative_gap'),
                    'surrogate_optimal': meta.get('surrogate_optimal'),
                    'model_load_seconds': meta.get('model_load_seconds', 0.0),
                    'tuning_seconds_this_run': 0.0}
                for key in ['solve_seconds', 'search_total_seconds', 'model_load_seconds']:
                    _finite(base[key], key, True)
                for policy in bc['protocol']['policies']:
                    key = (x.sha, method, seed, policy)
                    expected_keys.add(key)
                    if key not in indexed:
                        raise ValueError('Falta una réplica/política esperada.')
                    if solution['routes'] is None:
                        expected_row = {**base, 'policy': policy, 'evaluation_status': 'no_solution',
                                        **dict.fromkeys(e.SUMMARY_METRICS)}
                    else:
                        p.validate_routes(x, solution['routes'])  # sólo cobertura/capacidad, sin simulación
                        expected_bundles.add(f'{job}/{policy}')
                        result_dir = _safe_relative(directory, f'{job}/{policy}')
                        evaluated = e.read_json(result_dir / 'summary.json')
                        if any(evaluated.get(k) != v for k, v in {**context, 'policy': policy}.items()):
                            raise ValueError('Evaluación de otra identidad/política.')
                        qc = e.read_json(result_dir / 'validation.json')
                        if qc.get('status') != 'passed' or any(qc.get(k) is not True for k in
                                ['structural_valid', 'instance_unchanged', 'trace_matches_evaluator',
                                 'recourse_preserves_vehicle_assignment', 'cvar_independently_checked']):
                            raise ValueError('Evaluación sin trazas validadas.')
                        if (qc['scenarios'] != len(bc['scenarios'][x.sha])
                                or qc['event_rows'] != qc['expected_event_rows']
                                or qc['realized_route_rows'] != qc['expected_realized_route_rows']
                                or qc['event_rows'] != len(bc['scenarios'][x.sha]) * (x.n + sum(bool(r) for r in solution['routes']))
                                or qc['realized_route_rows'] != len(bc['scenarios'][x.sha]) * (x.n + 2 * len(x.capacities))
                                or qc['nominal_feasible'] != nominal['strict_feasible']):
                            raise ValueError('Conteos de evaluación inconsistentes.')
                        records = _read_scenarios(result_dir / 'scenario_metrics.csv',
                            {**context, 'policy': policy}, bc['scenarios'][x.sha])
                        if records[0]['used_vehicles'] != sum(bool(route) for route in solution['routes']):
                            raise ValueError('Vehículos evaluados distintos de las rutas persistidas.')
                        _check_aggregates(records, evaluated, bc['protocol']['alpha'])
                        for time_key in ['evaluation_seconds', 'trace_validation_seconds']:
                            _finite(evaluated[time_key], time_key, True)
                        scenario_rows.extend(records)
                        expected_row = {**base, **evaluated, 'evaluation_status': 'validated'}
                    if indexed[key] != expected_row:
                        raise ValueError('Reporte distinto del resultado persistido de un intento.')
    if set(indexed) != expected_keys or set(bundles) != expected_bundles:
        raise ValueError('Reporte o inventario con intentos faltantes/extra.')
    validation = inspected['validation']
    if (validation['summary_rows'] != len(rows)
            or validation['attempts'] != len(rows) // len(bc['protocol']['policies'])
            or validation['unsolved_policy_rows'] != sum(r['evaluation_status'] != 'validated' for r in rows)):
        raise ValueError('Resumen de validación no corresponde a los intentos.')
    return rows, scenario_rows, inspected


def _offline_costs(rows, drive_root, bank_id):
    """Una fila por manifiesto de entrenamiento; corpus docente en tabla separada."""
    groups = {}
    for row in rows:
        if row['method'] not in LEARNED:
            continue
        key = (LEARNED[row['method']], row['offline_cost_id'])
        groups.setdefault(key, []).append(row)
    costs, teachers = [], {}
    for (namespace, cost_id), records in sorted(groups.items()):
        methods = sorted({r['method'] for r in records})
        method = '|'.join(methods)
        if not cost_id or Path(cost_id).name != cost_id or cost_id in ('.', '..'):
            raise ValueError('Identidad offline inválida.')
        manifests = {r['offline_manifest_sha256'] for r in records}
        if len(manifests) != 1:
            raise ValueError('Un mismo costo offline tiene manifiestos diferentes.')
        selected = {}
        for record in records:
            model_key = (record['fold_id'], record['training_seed'], record['model_checkpoint_sha256'])
            value = {k: record[k] for k in ['training_seconds', 'validation_seconds']}
            if model_key in selected and selected[model_key] != value:
                raise ValueError('Costo contradictorio para el mismo modelo.')
            selected[model_key] = value
        output = {'method': method, 'shared_by_methods': '|'.join(methods),
                  'charged_once_across_ablations': True, 'offline_cost_id': cost_id,
                  'offline_manifest_sha256': next(iter(manifests)),
                  'selected_models': len(selected),
                  'selected_training_seconds': sum(v['training_seconds'] for v in selected.values()),
                  'selected_validation_seconds_included': sum(v['validation_seconds'] for v in selected.values()),
                  'manifest_training_seconds': None, 'manifest_models_trained': None,
                  'teacher_corpus_id': None, 'status': 'manifest_missing',
                  'scope': 'selected models deduplicated; manifest totals include all trained models',
                  'validation_is_part_of_training': True}
        index = Path(drive_root) / namespace / bank_id / cost_id / 'index'
        if not index.exists():
            costs.append(output)
            continue
        if _verify(index) != output['offline_manifest_sha256']:
            raise ValueError('Manifiesto de costo offline modificado.')
        manifest = e.read_json(index / 'offline_costs.json')
        models = e.read_json(index / 'models.json')
        if manifest['offline_cost_id'] != cost_id:
            raise ValueError('Inventario offline corresponde a otro entrenamiento.')
        available = {(m['fold']['fold_id'], m['training_seed'], m['model_checkpoint_sha256']): m for m in models}
        if len(available) != len(models) or not set(selected) <= set(available):
            raise ValueError('Modelos seleccionados ausentes/duplicados en inventario offline.')
        for key, values in selected.items():
            if any(values[k] != available[key][k] for k in values):
                raise ValueError('Costos por modelo distintos de manifiesto offline.')
        training = sum(_finite(m['training_seconds'], 'training_seconds', True) for m in models)
        validation = sum(_finite(m['validation_seconds'], 'validation_seconds', True) for m in models)
        if (manifest['models_trained'] != len(models)
                or not math.isclose(training, _finite(manifest['training_seconds_unique'], 'training_total', True))
                or not math.isclose(validation, _finite(manifest['validation_seconds_included_in_training'], 'validation_total', True))):
            raise ValueError('Totales offline no reconcilian inventario de modelos.')
        output.update(status='verified', manifest_training_seconds=training,
                      manifest_models_trained=len(models), teacher_corpus_id=manifest.get('teacher_corpus_id'))
        if output['teacher_corpus_id']:
            teacher_id = output['teacher_corpus_id']
            teacher = {'teacher_corpus_id': teacher_id, **manifest['teacher_costs_unique']}
            for field, value in teacher.items():
                if field != 'teacher_corpus_id':
                    _finite(value, field, True)
            if teacher_id in teachers and teachers[teacher_id] != teacher:
                raise ValueError('Costos contradictorios del mismo corpus docente.')
            teachers[teacher_id] = teacher
        costs.append(output)
    return costs, list(teachers.values())


def _coverage(rows, bc, config, selection):
    result = []
    for method in config['expected_methods']:
        selected = [r for r in rows if r['method'] == method]
        expected = len(bc['registry']) * len(bc['protocol']['solver_seeds']) * len(bc['protocol']['policies'])
        evaluated = sum(r['evaluation_status'] == 'validated' for r in selected)
        license_pending = sum(r['solver_status'] == 'license_pending' for r in selected)
        result.append({'method': method, 'run_id': selection.get(method),
            'expected_policy_rows': expected, 'recorded_policy_rows': len(selected),
            'evaluated_policy_rows': evaluated, 'license_pending_policy_rows': license_pending,
            'failed_policy_rows': len(selected) - evaluated - license_pending,
            'missing_policy_rows': expected - len(selected),
            'complete_evaluated_cohort': evaluated == expected})
    return result


def _report_text(coverage, scope, selection):
    lines = ['# Comparación 06 — piloto de desarrollo', '',
        'Resultados descriptivos de las ejecuciones seleccionadas. No hay pruebas de significancia, '
        'ranking poblacional ni afirmación de superioridad. Semillas y escenarios no aumentan el número '
        'de geometrías independientes. Los folds comparten entrenamiento y no son pruebas externas independientes.', '',
        'Costo: minutos de viaje ejecutado; no es el costo completo de recurso c+Q. '
        'CVaR se calcula por plan y después se promedia entre réplicas. '
        'Costo comparativo se condiciona a planes conservadoramente factibles; FR condicional requiere evaluación. '
        'joint_performance también informa viaje de todas las evaluaciones y los denominadores. Interpretar con cobertura.', '',
        '| Método | Evaluaciones | Esperadas | Fallos | Licencia pendiente | Faltantes |',
        '|---|---:|---:|---:|---:|---:|']
    for r in coverage:
        lines.append(f"| {r['method']} | {r['evaluated_policy_rows']} | {r['expected_policy_rows']} | "
                     f"{r['failed_policy_rows']} | {r['license_pending_policy_rows']} | {r['missing_policy_rows']} |")
    lines += ['', 'Las filas anteriores cuentan intento × política; las dos políticas no son muestras independientes.', '',
        f"Diferencias frente a {scope['reference_method']} sólo para cohortes completas en ambos métodos; no se emparejan "
        'semillas de solver como si representaran el mismo flujo aleatorio. Las diferencias de escenarios '
        'sí usan los CRN del banco. Las cotas Gurobi pertenecen exclusivamente al objetivo conservador.', '',
        'Tiempos de entrenamiento y validación se deduplican por modelo; validación ya está incluida '
        'en entrenamiento. Los costos docentes se deduplican por corpus. Un manifiesto ausente impide '
        'verificar su total y se identifica como tal. No se amortizan costos offline sobre el piloto.', '',
        'Las comparaciones de tiempo requieren considerar el hardware de cada ejecución (input_environments.json). '
        '06 verifica hashes, identidades y agregados; no vuelve a simular las rutas.', '',
        'Selección de runs (no basada automáticamente en fecha o rendimiento):',
        '```json', json.dumps(selection, indent=2), '```', '',
        'Tablas adicionales: joint_performance informa éxito sobre todos los intentos y cotas '
        'de desconocimiento para faltantes; no son intervalos de confianza. paired_geometry_costs '
        'contrasta costos sólo en geometrías con todas las semillas factibles en ambos métodos y '
        'conserva el número y motivo de exclusiones. No existe una penalización de costo inventada. '
        'risk_diagnostics describe varianza, dispersión y exceso CVaR; recourse_differences contrasta '
        'causal_edd contra fixed_order sobre el mismo plan y CRN. budget_audit deduplica las políticas '
        'para no multiplicar tiempo de búsqueda. La comparación POMO/sin entrenar y HBG/sin feromona '
        'se fija antes de observar resultados. No se ha implementado una ejecución confirmatoria: '
        'inference_readiness enumera explícitamente los requisitos pendientes.', '',
        'Las figuras se exportan en PNG (300 dpi), PDF y SVG; costos condicionados y ausencia de '
        'réplicas permanecen visibles. No se ordenan métodos por desempeño.', '',
        'Alcance estadístico:', '```json', json.dumps(scope, indent=2, ensure_ascii=False), '```', '']
    return '\n'.join(lines)


def _discussion_table_text(tables):
    """Publication-oriented descriptive table with denominators beside costs."""
    output = ['# Tablas para discusión · piloto descriptivo', '',
        'Viaje y CVaR se condicionan a planes conservadoramente factibles. Éxito incluye '
        'todos los intentos previstos; si falta información se muestra un intervalo de '
        'desconocimiento, no un intervalo de confianza. Un guion indica ausencia de medición.', '']
    groups = {}
    for row in tables['joint_performance']:
        groups.setdefault((row['instance_id'], row['n_customers'], row['policy']), []).append(row)
    def number(value):
        return '—' if value is None else f'{value:.2f}'
    for (instance_id, n, policy), rows in groups.items():
        output += [f'## {instance_id} · n={n} · {policy}', '',
            '| Método | Evaluados / previstos | Factibles conservadores / previstos | Viaje (min) | CVaR por plan (min) | Éxito total (%) | Fallos | Faltantes/licencia |',
            '|---|---:|---:|---:|---:|---:|---:|---:|']
        for r in rows:
            lo, hi = 100 * r['scenario_success_lower_bound'], 100 * r['scenario_success_upper_bound']
            success = number(lo) if r['scenario_success_per_scheduled_attempt'] is not None else f'[{lo:.2f}, {hi:.2f}]'
            output.append(f"| {r['method']} | {r['evaluated_attempts']}/{r['expected_attempts']} | "
                f"{r['conservative_feasible_attempts']}/{r['expected_attempts']} | "
                f"{number(r['mean_travel_conditional_conservative_feasible'])} | "
                f"{number(r['cvar_travel_conditional_conservative_feasible'])} | {success} | "
                f"{r['failed_attempts']} | {r['missing_attempts']}/{r['license_pending_attempts']} |")
        output.append('')
    output += ['No hay selección de mejor semilla, ranking global ni prueba confirmatoria. '
               'Cada geometría cuenta una vez; escenarios y semillas no aumentan N.', '']
    return '\n'.join(output)


def _solution_diagnostics(solution):
    """Flatten recorded metadata without imputing iterations or attributing unknown stops."""
    meta = solution['metadata']
    raw = solution['status']
    normalized = ('time_budget_reached' if raw == 'interrupted' and
                  meta.get('budget_termination_requested') is True else raw)
    attempts, complete = meta.get('attempts'), meta.get('completed_decodes')
    return {'solver_status': raw, 'solver_status_normalized': normalized,
        'status_normalization_basis': 'recorded_budget_callback' if normalized != raw else 'original_status',
        **{key: meta.get(key) for key in (
            'termination', 'attempts', 'completed_decodes', 'failed_decodes', 'interrupted_decodes',
            'iterations', 'completed_iterations', 'constructed_solutions', 'initial_attempts',
            'evaluated_neighbors', 'accepted_moves', 'constructive_seed_count', 'neural_complete_count',
            'best_candidate_generation_source', 'implementation', 'construction',
            'inference_device', 'model_config', 'budget_termination_requested',
            'solution_origin', 'returned_paid_edd_fallback', 'routing_assignment_found')},
        'decode_completion_fraction': complete / attempts if type(attempts) is int and attempts > 0
            and type(complete) is int and 0 <= complete <= attempts else None,
        'solver_metadata_json': json.dumps(meta, sort_keys=True, ensure_ascii=False)}


def run_comparison(bank, method_runs_root, output_root, config, selection=None,
                   local_root=None, validation_artifacts=()):
    """Exporta un análisis sellado reproducible; nunca modifica los resultados fuente."""
    _require_colab()
    bc = _bank_context(bank, config)
    inventory = discover_runs(bank, method_runs_root, config)
    chosen = select_runs(inventory, config, selection)
    rows, scenarios, inputs, environments, diagnostics, sources_by_method = [], [], {}, {}, [], {}
    for run_id in sorted(set(chosen.values())):
        directory = _safe_relative(method_runs_root, run_id)
        summary, raw, inspected = _load_run(directory, bc)
        selected_methods = {m for m, selected_id in chosen.items() if selected_id == run_id}
        sources_by_method.update({m: {**inspected['plan']['source_hashes'],
            '__environment_sha256': p.digest(e.read_json(directory / 'meta/environment.json'))}
            for m in selected_methods})
        rows.extend(r for r in summary if r['method'] in selected_methods)
        scenarios.extend(r for r in raw if r['method'] in selected_methods)
        inputs[run_id] = {k: inspected[k] for k in ['report_sha256', 'meta_sha256']}
        environments[run_id] = e.read_json(directory / 'meta/environment.json')
        for method in sorted(selected_methods):
            for x in bc['instances']:
                for seed in bc['protocol']['solver_seeds']:
                    solution = e.read_json(directory / method / f'n{x.n:04d}' / f'seed_{seed}' / 'solution/solution.json')
                    meta = solution['metadata']
                    diagnostics.append({'run_id': run_id, 'method': method, 'solver_seed': seed,
                        'instance_sha256': x.sha, 'n_customers': x.n,
                        'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                        **_solution_diagnostics(solution)})
    contrast_audit = _controlled_contrast_audit(rows, diagnostics, bc['registry'],
        bc['protocol']['solver_seeds'], config['method_contract']['methods'], sources_by_method)
    tables = stats.build_tables(rows, scenarios, bc['registry'], bc['protocol']['policies'],
                               bc['protocol']['solver_seeds'], config, contrast_audit=contrast_audit)
    costs, teachers = _offline_costs(rows, Path(method_runs_root).parent.parent, bc['bank_id'])
    tables.update(offline_costs=costs, teacher_costs=teachers, solution_diagnostics=diagnostics)
    tables['method_provenance'] = [
        {'method': method, 'run_id': run_id, 'comparison_id': bc['comparison_id'],
         'objective': bc['protocol']['objective'],
         'search_budget_seconds': bc['protocol']['search_seconds'],
         'algorithm_config_json': json.dumps(e.read_json(_safe_relative(method_runs_root, run_id) / 'meta/plan.json')['config']['methods'][method], sort_keys=True),
         'role': config.get('method_roles', {}).get(method, 'comparator'),
         'environment_file': 'input_environments.json', 'environment_run_key': run_id,
         'same_budget_does_not_imply_same_hardware': True}
        for method, run_id in chosen.items()]
    coverage = _coverage(rows, bc, config, chosen)
    sources = {name: p.file_hash(Path(__file__).with_name(name)) for name in
               ['comparison.py', 'comparison_stats.py', 'experiments.py', 'search.py']}
    identity = {'schema_version': SCHEMA, 'config': config, 'comparison_id': bc['comparison_id'],
                'selected_runs': chosen, 'inputs': inputs, 'source_hashes': sources,
                'offline_costs_sha256': p.digest(costs), 'teacher_costs_sha256': p.digest(teachers),
                'validation_hashes': {Path(f).name: p.file_hash(f) for f in validation_artifacts}}
    analysis_id = 'comparison06_' + p.digest(identity)[:20]
    remote = Path(output_root) / analysis_id / 'report'
    local = Path(local_root or '/content/comparison06') / analysis_id / 'report'
    if local.resolve() == remote.resolve():
        raise ValueError('Staging local y destino Drive deben ser distintos.')
    for destination in [local, remote]:
        if any(destination.resolve().is_relative_to(Path(source).resolve())
               for source in [bank, method_runs_root]):
            raise ValueError('Las salidas deben quedar fuera del banco y las corridas fuente.')
    if not e._recover(local, remote):
        local.mkdir(parents=True, exist_ok=True)
        e.write_json(local / 'identity.json', identity)
        e.write_json(local / 'input_inventory.json', inventory)
        e.write_json(local / 'input_environments.json', environments)
        e.write_json(local / 'environment.json', p.environment_report())
        e.write_json(local / 'registry.json', bc['registry'])
        e.write_csv(local / 'registry.csv', bc['registry'])
        e.write_json(local / 'common_protocol.json', bc['protocol'])
        e.write_json(local / 'analysis_export_contract.json', {
            'schema_version': 'analysis-export-08-v1', 'comparison_schema': SCHEMA,
            'bank_id': bc['bank_id'], 'bank_manifest_sha256': bc['bank_hash'],
            'comparison_id': bc['comparison_id'], 'role': config['role'],
            'analysis_mode': config['analysis_mode'], 'unit_of_independence': 'geometry_group_sha256',
            'policies': bc['protocol']['policies'], 'solver_seeds': bc['protocol']['solver_seeds'],
            'expected_methods': config['expected_methods'], 'reference_method': config['reference_method'],
            'source_selection': chosen, 'source_selection_is_best_run_search': False,
            'required_files': ['identity.json', 'validation.json', 'attempts.csv', 'attempts.json',
                'registry.csv', 'registry.json', 'common_protocol.json', 'tables.json',
                'joint_performance.csv', 'case_summary.csv', 'paired_geometry_costs.csv',
                'paired_geometry_success.csv', 'gaps.csv', 'solution_diagnostics.csv',
                'statistical_scope.json', 'input_environments.json']})
        e.write_json(local / 'attempts.json', rows)
        e.write_csv(local / 'attempts.csv', rows)
        e.write_json(local / 'coverage.json', coverage)
        e.write_csv(local / 'coverage.csv', coverage)
        e.write_json(local / 'tables.json', tables)
        for name, data in tables.items():
            if isinstance(data, list):
                e.write_csv(local / f'{name}.csv', data, None if data else ['status'])
        e.write_json(local / 'statistical_scope.json', tables['statistical_scope'])
        for name in sources:
            destination = local / 'code' / name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(Path(__file__).with_name(name), destination)
        for source in map(Path, validation_artifacts):
            destination = local / 'contract_validation' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        stats.save_plots(tables, local / 'figures')
        (local / 'report.md').write_text(_report_text(coverage, tables['statistical_scope'], chosen), encoding='utf-8')
        (local / 'discussion_tables.md').write_text(_discussion_table_text(tables), encoding='utf-8')
        # Revalidar los bundles fuente detecta cambios durante el análisis.
        for run_id, hashes in inputs.items():
            directory = _safe_relative(method_runs_root, run_id)
            for part in ['meta', 'report']:
                if _verify(directory / part) != hashes[f'{part}_sha256']:
                    raise ValueError('Un resultado fuente cambió durante el análisis.')
            for relative, digest in e.read_json(directory / 'report/bundles.json').items():
                if _verify(_safe_relative(directory, relative)) != digest:
                    raise ValueError('Un bundle fuente cambió durante el análisis.')
        if _verify(bank) != bc['bank_hash']:
            raise ValueError('El banco cambió durante el análisis.')
        final_costs, final_teachers = _offline_costs(rows, Path(method_runs_root).parent.parent, bc['bank_id'])
        if final_costs != costs or final_teachers != teachers:
            raise ValueError('Los manifiestos offline cambiaron durante el análisis.')
        e.write_json(local / 'validation.json', {'status': 'completed', 'bank_unchanged': True,
            'source_results_unchanged': True, 'offline_manifests_unchanged': True,
            'scenario_aggregates_reconciled': True,
            'no_simulations_or_solvers_run': True, 'ready_for_confirmatory_comparison': False,
            'selected_runs': chosen, 'summary_rows': len(rows), 'scenario_rows': len(scenarios)})
        e._commit(local, remote)
    if e.read_json(local / 'identity.json') != identity:
        raise ValueError('Reporte recuperado con identidad distinta.')
    files = [str(f.relative_to(remote)) for f in sorted(remote.rglob('*')) if f.is_file()]
    return {'report_dir': str(remote), 'comparison_run_id': analysis_id, 'coverage': coverage,
            'statistical_scope': tables['statistical_scope'], 'files': files, 'tables': tables}


Writing svrplab/comparison.py


In [9]:
import importlib
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))
for name in list(sys.modules):
    if name == 'svrplab' or name.startswith('svrplab.'):
        del sys.modules[name]
importlib.invalidate_caches()
from svrplab import pilot as p
from svrplab import experiments as exp
from svrplab import comparison as cmp
cmp.validate_config(COMPARISON_CONFIG)
import pandas as pd
from IPython.display import display, Image, Markdown
METHOD_RUNS_ROOT = DRIVE_ROOT / 'method_runs' / BANK_ID
OUTPUT_ROOT = DRIVE_ROOT / 'comparison_runs' / BANK_ID
LOCAL_ROOT = WORK / 'comparison_runs' / BANK_ID
if not METHOD_RUNS_ROOT.is_dir():
    print('Aún no hay directorio de exportaciones de 01–05; se pueden ejecutar contratos, pero la selección requerirá corridas compatibles.')
print('Corridas de entrada:', METHOD_RUNS_ROOT)
print('Reportes de salida:', OUTPUT_ROOT)


Corridas de entrada: /content/drive/MyDrive/TESIS-EHBG-FACS/method_runs/pilot00_280553a33c2ab594
Reportes de salida: /content/drive/MyDrive/TESIS-EHBG-FACS/comparison_runs/pilot00_280553a33c2ab594


## 4. Pruebas de contrato, únicamente en Colab
Las pruebas verifican integridad, selección, compatibilidad, cobertura y agregación
estadística sobre fixtures pequeños. No convierten esos fixtures en observaciones
del banco. Se conserva la evidencia en Drive aunque fallen; un fallo detiene el análisis.
Estas pruebas no se han ejecutado localmente.


In [10]:
BUNDLED_CONFIGS = {'accepted_bank_00.json': {'bank_id': 'pilot00_280553a33c2ab594', 'protocol_version': 'pilot-00-v4', 'manifest_sha256': 'd8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02', 'acceptance_status': 'ready_for_method_development', 'role': 'development_only', 'validation_run_id': '20260917T021417Z_665a3a34', 'evidence': 'docs/NOTEBOOK_00_VALIDATION.md'}, 'methods_01_02.json': {'schema_version': 'methods-v2', 'role': 'development_only', 'objective': 'conservative_upper_travel_minutes_v1', 'time_scale': 1000, 'seeds': [11, 29, 47], 'search_seconds': 120.0, 'scenario_split': 'evaluation', 'export_all_scenario_events': True, 'methods': {'aco': {'max_iterations': None, 'ants': 20, 'alpha': 1.0, 'beta': 2.0, 'evaporation': 0.1, 'q0': 0.2, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'tabu': {'max_iterations': None, 'neighborhood_samples': 80, 'tenure': 15, 'initial_restarts': 4, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'gurobi': {'threads': 1, 'mip_gap': 0.0}, 'am_supervised': {'n_samples': 16, 'permute_samples': False, 'completion_search_nodes': 2048}, 'pomo': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'hbg_facs': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'edd': {}, 'ortools_routing': {'capacity_scale': 1000}, 'pomo_untrained': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'random_masked': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'hbg_no_pheromone': {'ants': 16, 'iterations': 100, 'alpha': 0.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'hbg_untrained': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}}}, 'comparison_06.json': {'schema_version': 'comparison-06-v2', 'role': 'development_only', 'analysis_mode': 'descriptive_only', 'expected_methods': ['gurobi', 'aco', 'tabu', 'am_supervised', 'pomo', 'hbg_facs', 'edd', 'ortools_routing', 'pomo_untrained', 'random_masked', 'hbg_no_pheromone', 'hbg_untrained'], 'reference_method': 'gurobi', 'metrics': ['scenario_strict_feasibility', 'mean_travel_minutes', 'cvar_travel_minutes', 'mean_lateness_minutes', 'mean_depot_overtime_minutes', 'search_total_seconds'], 'accepted_bank': {'bank_id': 'pilot00_280553a33c2ab594', 'protocol_version': 'pilot-00-v4', 'manifest_sha256': 'd8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02', 'acceptance_status': 'ready_for_method_development', 'role': 'development_only', 'validation_run_id': '20260917T021417Z_665a3a34', 'evidence': 'docs/NOTEBOOK_00_VALIDATION.md'}, 'method_contract': {'schema_version': 'methods-v2', 'role': 'development_only', 'objective': 'conservative_upper_travel_minutes_v1', 'time_scale': 1000, 'seeds': [11, 29, 47], 'search_seconds': 120.0, 'scenario_split': 'evaluation', 'export_all_scenario_events': True, 'methods': {'aco': {'max_iterations': None, 'ants': 20, 'alpha': 1.0, 'beta': 2.0, 'evaporation': 0.1, 'q0': 0.2, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'tabu': {'max_iterations': None, 'neighborhood_samples': 80, 'tenure': 15, 'initial_restarts': 4, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'gurobi': {'threads': 1, 'mip_gap': 0.0}, 'am_supervised': {'n_samples': 16, 'permute_samples': False, 'completion_search_nodes': 2048}, 'pomo': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'hbg_facs': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'edd': {}, 'ortools_routing': {'capacity_scale': 1000}, 'pomo_untrained': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'random_masked': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'hbg_no_pheromone': {'ants': 16, 'iterations': 100, 'alpha': 0.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'hbg_untrained': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}}}, 'controlled_contrasts': [{'method': 'pomo', 'reference': 'pomo_untrained'}, {'method': 'pomo', 'reference': 'random_masked'}, {'method': 'hbg_facs', 'reference': 'hbg_no_pheromone'}, {'method': 'hbg_facs', 'reference': 'hbg_untrained'}, {'method': 'hbg_facs', 'reference': 'ortools_routing'}, {'method': 'hbg_facs', 'reference': 'edd'}], 'method_roles': {'gurobi': 'comparator', 'aco': 'comparator', 'tabu': 'comparator', 'am_supervised': 'exploratory_secondary', 'pomo': 'comparator', 'hbg_facs': 'proposed_declared_adaptation', 'edd': 'comparator', 'ortools_routing': 'comparator', 'pomo_untrained': 'ablation', 'random_masked': 'ablation', 'hbg_no_pheromone': 'ablation', 'hbg_untrained': 'ablation'}, 'future_confirmatory_design': {'enabled': False, 'minimum_independent_test_geometries': None, 'sample_size_justification': None, 'primary_estimand': 'scenario_success_per_scheduled_attempt', 'planning_notebook': '07_auditoria_diseno.ipynb', 'status': 'requires_separate_locked_protocol_and_independent_test_bank'}, 'report_revision': 'final_development_descriptive_v1', 'reference_selection_note': 'Gurobi elegido tras el diagnóstico de desarrollo: revisión descriptiva declarada, no hipótesis confirmatoria preespecificada. Se promedian todas las réplicas previstas.'}}

(RUNTIME / 'configs').mkdir(exist_ok=True)
for filename, value in BUNDLED_CONFIGS.items():
    (RUNTIME / 'configs' / filename).write_text(json.dumps(value, indent=2, ensure_ascii=False) + '\n')


In [11]:
%%writefile tests/test_comparison_stats.py
"""Controles de 06, exclusivamente en Google Colab; localmente sólo AST.

Fixtures manuales sin generar bancos, ejecutar métodos ni simular escenarios.
"""
import copy
from pathlib import Path
import sys

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import comparison_stats as stats


SEEDS = [11, 22]
SCENARIOS = [18446744073709551613, 18446744073709551615]


def config():
    return {'schema_version': 'comparison-06-v2', 'role': 'development_only',
            'analysis_mode': 'descriptive_only', 'expected_methods': list(stats.METHODS),
            'reference_method': 'aco', 'metrics': list(stats.METRICS)}


def instance(sha='case-a', geometry='geometry-a', n=10):
    return {'instance_id': sha, 'instance_sha256': sha,
            'geometry_group_sha256': geometry, 'n_customers': n}


def summary(method, seed, cost=20., *, identity=None, status='validated', solver_status='feasible',
            feasible=True, bound=None, optimal=False):
    identity = identity or instance()
    valid = status == 'validated'
    return {**identity, 'method': method, 'solver_seed': seed, 'policy': 'fixed_order',
            'evaluation_status': status, 'solver_status': solver_status,
            'structural_valid': valid, 'conservative_feasible': feasible if valid else None,
            'scenarios': 2 if valid else None,
            'scenario_strict_feasibility': .5 if valid else None,
            'mean_travel_minutes': cost if valid else None,
            'cvar_travel_minutes': cost + 2 if valid else None,
            'mean_lateness_minutes': 2. if valid else None,
            'mean_depot_overtime_minutes': 0. if valid else None,
            'search_total_seconds': 7., 'search_objective_minutes': cost if valid else None,
            'best_bound': bound, 'surrogate_optimal': optimal}


def scenarios(row):
    return [{key: row[key] for key in ('instance_id', 'instance_sha256', 'geometry_group_sha256',
                                      'n_customers', 'method', 'solver_seed', 'policy')} | {
                'scenario_seed': str(seed), 'travel_minutes': row['mean_travel_minutes'] + offset,
                'strict_feasible': offset < 0, 'lateness_minutes': 0. if offset < 0 else 4.,
                'depot_overtime_minutes': 0.,
            } for seed, offset in zip(SCENARIOS, [-2., 2.])]


def build(rows, *, raw=None, registry=None, contrast_audit=None):
    return stats.build_tables(rows, raw or [], registry or [instance()], ['fixed_order'], SEEDS, config(),
                              contrast_audit=contrast_audit)


def select(tables, table, *, method='aco', metric='mean_travel_minutes', **criteria):
    matching = [row for row in tables[table] if row.get('method') == method
                and (metric is None or row.get('metric') == metric)
                and all(row.get(key) == value for key, value in criteria.items())]
    assert len(matching) == 1
    return matching[0]


def test_missing_and_license_pending_remain_explicit_nulls():
    rows = [summary('aco', 11, 20), summary('aco', 22, 30),
            summary('gurobi', 11, status='no_solution', solver_status='license_pending')]
    tables = build(rows)
    missing = select(tables, 'case_summary', method='pomo')
    assert missing['expected'] == missing['missing'] == 2
    assert missing['evaluated'] == missing['metric_available'] == 0
    assert missing['mean_conditional'] is None and not missing['cohort_complete']
    licensed = select(tables, 'case_summary', method='gurobi')
    assert licensed['license_pending'] == licensed['missing'] == 1
    assert licensed['failed'] == licensed['evaluated'] == 0 and licensed['mean_conditional'] is None
    runtime = select(tables, 'case_summary', method='gurobi', metric='search_total_seconds')
    assert runtime['mean_conditional'] == 7. and runtime['metric_available'] == 1
    assert runtime['conditioning'] == 'recorded_attempts_with_runtime'
    assert len(tables['case_summary']) == len(stats.METHODS) * len(stats.METRICS)


def test_failure_costs_are_conditional_but_runtime_covers_observed_attempts():
    failed = summary('pomo', 22, status='no_solution', solver_status='no_solution')
    failed['search_total_seconds'] = 13.
    # A stale cost on a failed attempt must never enter conditional evaluation means.
    failed['mean_travel_minutes'] = 9999.
    tables = build([summary('pomo', 11, 10), failed])
    cost = select(tables, 'case_summary', method='pomo')
    assert cost['mean_conditional'] == 10 and cost['failed'] == 1
    assert cost['evaluated_fraction'] == .5 and not cost['cohort_complete']
    runtime = select(tables, 'case_summary', method='pomo', metric='search_total_seconds')
    assert runtime['mean_conditional'] == 10 and runtime['metric_available'] == 2
    assert not runtime['cohort_complete']


def test_geometry_averages_instances_after_replicas_without_pseudoreplication():
    a, b, c = instance('a', 'same'), instance('b', 'same'), instance('c', 'other', n=20)
    rows = [summary('aco', 11, 10, identity=a), summary('aco', 22, 30, identity=a),
            summary('aco', 11, 100, identity=b)]
    tables = build(rows, registry=[a, b, c])
    geometry = select(tables, 'geometry_summary', geometry_group_sha256='same')
    assert geometry['mean_conditional'] == 60  # mean(mean(10,30), mean(100)).
    assert geometry['expected_instances'] == geometry['instances_with_measurement'] == 2
    assert geometry['expected'] == 4 and geometry['evaluated'] == 3 and geometry['missing'] == 1
    assert geometry['independent_geometry_count'] == 1
    assert geometry['fully_evaluated_instances'] == 1 and not geometry['cohort_complete']
    assert tables['statistical_scope']['independent_geometry_count'] == 2
    assert tables['statistical_scope']['instance_count'] == 3
    assert tables['statistical_scope']['no_pvalues']
    assert not tables['statistical_scope']['ready_for_confirmatory']


def test_pairwise_requires_complete_validated_cohorts_for_both_methods():
    rows = [summary('aco', 11, 20), summary('aco', 22, 40), summary('pomo', 11, 5)]
    tables = build(rows)
    row = select(tables, 'pairwise', method='pomo')
    assert row['status'] == 'incomplete_cohort'
    assert row['delta_mean'] is row['ratio_percent'] is row['method_mean'] is None
    assert row['method_evaluated'] == 1 and row['reference_evaluated'] == 2
    assert not row['solver_seed_pairing']
    rows.append(summary('pomo', 22, 15))
    complete = select(build(rows), 'pairwise', method='pomo')
    assert complete['delta_mean'] == -20
    assert complete['ratio_percent'] == pytest.approx(-100 * 20 / 30)
    assert complete['status'] == 'complete'


def test_zero_reference_has_difference_but_no_ratio():
    rows = [summary(method, seed, 0 if method == 'aco' else 5)
            for method in ('aco', 'pomo') for seed in SEEDS]
    for row in rows:
        row['scenario_strict_feasibility'] = 0. if row['method'] == 'aco' else 1.
    tables = build(rows)
    cost = select(tables, 'pairwise', method='pomo')
    assert cost['delta_mean'] == 5 and cost['ratio_percent'] is None
    assert cost['ratio_status'] == 'zero_reference'
    fr = select(tables, 'pairwise', method='pomo', metric='scenario_strict_feasibility')
    assert fr['delta_percentage_points'] == 100. and fr['ratio_percent'] is None


def test_cvar_is_mean_of_per_plan_values_not_cvar_of_pooled_costs():
    rows = [summary('aco', 11, 10), summary('aco', 22, 30)]
    rows[0]['cvar_travel_minutes'], rows[1]['cvar_travel_minutes'] = 12., 32.
    tables = build(rows, raw=[scenario for row in rows for scenario in scenarios(row)])
    row = select(tables, 'case_summary', metric='cvar_travel_minutes')
    assert row['mean_conditional'] == 22.
    assert row['cvar_scope'] == 'mean_of_per_plan_empirical_cvar'
    assert all(row['metric'] != 'cvar_travel_minutes' for row in tables['scenario_differences'])


def test_crn_differences_first_average_solver_replicas_and_preserve_64bit_seeds():
    rows = [summary('aco', 11, 10), summary('aco', 22, 30),
            summary('pomo', 11, 14), summary('pomo', 22, 18)]
    raw = [scenario for row in rows for scenario in scenarios(row)]
    tables = build(rows, raw=raw)
    differences = [row for row in tables['scenario_differences'] if row['metric'] == 'mean_travel_minutes']
    assert len(differences) == 2
    assert [row['scenario_seed'] for row in differences] == [str(seed) for seed in SCENARIOS]
    assert [row['method_replica_mean'] for row in differences] == [14., 18.]
    assert [row['reference_replica_mean'] for row in differences] == [18., 22.]
    assert all(row['delta_mean'] == -4. and row['replicas_per_method'] == 2 for row in differences)
    assert all(not row['solver_seed_pairing'] and not row['independent_population_observation'] for row in differences)


def test_crn_never_selects_only_successful_solver_seeds():
    rows = [summary('aco', 11, 10), summary('aco', 22, 30), summary('pomo', 11, 14)]
    tables = build(rows, raw=[scenario for row in rows for scenario in scenarios(row)])
    assert tables['scenario_differences'] == []


def test_truncated_mismatched_or_duplicate_scenarios_rejected():
    rows = [summary('aco', 11, 10), summary('aco', 22, 30)]
    raw = [scenario for row in rows for scenario in scenarios(row)]
    with pytest.raises(ValueError, match='incompleta'):
        build(rows, raw=raw[:-1])
    mismatched = copy.deepcopy(raw)
    mismatched[-1]['scenario_seed'] = '77'
    with pytest.raises(ValueError, match='CRN'):
        build(rows, raw=mismatched)
    with pytest.raises(ValueError, match='duplicado'):
        build(rows, raw=[*raw, raw[0]])


def test_gap_uses_best_same_problem_bound_including_timeout_without_solution():
    a, b = instance('a', 'geo-a'), instance('b', 'geo-b')
    bound_only = summary('gurobi', 22, identity=a, status='no_solution', solver_status='time_limit', bound=80.)
    rows = [summary('gurobi', 11, 120, identity=a, bound=70.), bound_only,
            summary('aco', 11, 100, identity=a), summary('aco', 22, 120, identity=a),
            summary('gurobi', 11, 900, identity=b, bound=800.)]
    tables = build(rows, registry=[a, b])
    row = select(tables, 'gaps', method='aco', metric=None, instance_sha256='a', solver_seed=11)
    assert row['best_gurobi_lower_bound_minutes'] == 80 and row['gap_upperbound_pct'] == 20.
    assert row['bound_source_solver_seed'] == 22
    assert not row['gurobi_surrogate_optimum_validated']
    assert not row['numerical_equality_is_optimality_proof']
    missing = select(tables, 'gaps', method='pomo', metric=None, instance_sha256='a', solver_seed=11)
    assert missing['gap_upperbound_pct'] is None


def test_gap_excludes_nonconservative_candidate_and_inconsistent_lower_bound():
    rows = [summary('gurobi', 11, 100, bound=90.), summary('aco', 11, 80),
            summary('aco', 22, 100, feasible=False)]
    tables = build(rows)
    inconsistent = select(tables, 'gaps', metric=None, solver_seed=11)
    assert inconsistent['status'] == 'bound_exceeds_candidate_inconsistent'
    assert inconsistent['gap_upperbound_pct'] is None
    rejected = select(tables, 'gaps', metric=None, solver_seed=22)
    assert rejected['candidate_cost_minutes'] is None and rejected['gap_upperbound_pct'] is None


def test_zero_cost_and_numeric_equality_do_not_fabricate_optimality():
    rows = [summary('gurobi', 11, 100, bound=100. + 1e-8), summary('aco', 11, 100),
            summary('aco', 22, 0.)]
    tables = build(rows)
    equal = select(tables, 'gaps', metric=None, solver_seed=11)
    assert equal['gap_upperbound_pct'] == 0 and equal['numerical_equality_clipped_to_zero']
    assert not equal['gurobi_surrogate_optimum_validated']
    assert not equal['candidate_surrogate_optimal_validated']
    assert not equal['numerical_equality_is_optimality_proof']
    zero_rows = [summary('gurobi', 11, 0, bound=0.), summary('aco', 11, 0.)]
    zero = select(build(zero_rows), 'gaps', metric=None, solver_seed=11)
    assert zero['status'] == 'zero_candidate_ratio_undefined' and zero['gap_upperbound_pct'] is None


def test_only_validated_gurobi_surrogate_optimal_flag_supports_optimal_reference():
    rows = [summary('gurobi', 11, 80, bound=80., optimal=True), summary('aco', 11, 100)]
    tables = build(rows)
    exact = select(tables, 'gaps', method='gurobi', metric=None, solver_seed=11)
    assert exact['gurobi_surrogate_optimum_validated'] and exact['candidate_surrogate_optimal_validated']
    heuristic = select(tables, 'gaps', metric=None, solver_seed=11)
    assert heuristic['gurobi_surrogate_optimum_validated'] and not heuristic['candidate_surrogate_optimal_validated']
    assert heuristic['gap_upperbound_pct'] == 20
    rows[0]['best_bound'] = 60.
    inconsistent = select(build(rows), 'gaps', method='gurobi', metric=None, solver_seed=11)
    assert not inconsistent['gurobi_surrogate_optimum_validated']
    assert not inconsistent['candidate_surrogate_optimal_validated']


def test_duplicate_summary_unknown_cohort_and_float_seeds_rejected():
    row = summary('aco', 11)
    with pytest.raises(ValueError, match='duplicada'):
        build([row, row])
    other = copy.deepcopy(row)
    other['solver_seed'] = 99
    with pytest.raises(ValueError, match='cohorte'):
        build([other])
    other['solver_seed'] = 11.
    with pytest.raises(ValueError, match='enteros exactos'):
        build([other])


@pytest.mark.parametrize('change', [{'analysis_mode': 'confirmatory'}, {'role': 'final_test'},
                                    {'ready_for_confirmatory': True}, {'reference_method': 'pomo'},
                                    {'expected_methods': ['aco']}, {'metrics': ['mean_travel_minutes']}])
def test_protocol_blocks_confirmatory_or_posthoc_scope(change):
    with pytest.raises(ValueError):
        stats.build_tables([], [], [instance()], ['fixed_order'], SEEDS, config() | change)


def test_missing_metric_is_null_and_suppresses_that_pairwise_metric():
    rows = [summary(method, seed) for method in ('aco', 'pomo') for seed in SEEDS]
    rows[-1]['cvar_travel_minutes'] = None
    tables = build(rows)
    row = select(tables, 'case_summary', method='pomo', metric='cvar_travel_minutes')
    assert row['evaluated'] == 2 and row['metric_available'] == 1 and not row['cohort_complete']
    assert select(tables, 'pairwise', method='pomo', metric='cvar_travel_minutes')['delta_mean'] is None
    assert select(tables, 'pairwise', method='pomo')['status'] == 'complete'


def test_plots_export_one_nonempty_file_per_geometry_policy(tmp_path):
    pytest.importorskip('matplotlib')
    tables = build([summary('aco', 11), summary('aco', 22)])
    filenames = stats.save_plots(tables, tmp_path)
    assert len(filenames) == 6
    assert {Path(name).suffix for name in filenames} == {'.png', '.pdf', '.svg'}
    assert all(Path(name).name == name and (tmp_path / name).stat().st_size > 0 for name in filenames)


def test_joint_success_retains_no_solution_and_never_imputes_costs():
    failed = summary('pomo', 22, status='no_solution', solver_status='no_solution')
    tables = build([summary('pomo', 11, 10), failed])
    row = select(tables, 'joint_performance', method='pomo', metric=None)
    assert row['scenario_success_per_scheduled_attempt'] == .25
    assert row['scenario_success_lower_bound'] == row['scenario_success_upper_bound'] == .25
    assert row['mean_travel_conditional_evaluated'] == 10.
    assert row['conservative_feasible_attempts'] == 1
    assert row['expected_attempts'] == 2 and row['failed_attempts'] == 1


def test_missing_and_evaluation_error_are_unknown_not_zero_success():
    row = select(build([summary('pomo', 11)]), 'joint_performance', method='pomo', metric=None)
    assert row['scenario_success_per_scheduled_attempt'] is None
    assert row['scenario_success_lower_bound'] == .25 and row['scenario_success_upper_bound'] == .75
    assert not row['success_bounds_are_confidence_intervals']
    error = summary('pomo', 22, status='evaluation_error')
    error['structural_valid'] = True
    row = select(build([summary('pomo', 11), error]), 'joint_performance', method='pomo', metric=None)
    assert row['scenario_success_unresolved_attempts'] == 1


def test_paired_geometry_costs_require_every_seed_and_instance_of_both_methods():
    a, b = instance('a', 'same'), instance('b', 'same')
    rows = [summary(m, seed, identity=x) for m in ('aco', 'pomo') for seed in SEEDS for x in (a, b)]
    rows[-1]['conservative_feasible'] = False
    tables = build(rows, registry=[a, b])
    row = select(tables, 'paired_geometry_costs', method='pomo', reference_method='aco')
    assert row['expected_attempts_per_method'] == 4
    assert row['common_feasible_geometry_count'] == 0 and row['delta_mean'] is None
    rows[-1]['conservative_feasible'] = True
    row = select(build(rows, registry=[a, b]), 'paired_geometry_costs', method='pomo', reference_method='aco')
    assert row['common_feasible_geometry_count'] == row['expected_geometry_count'] == 1
    assert row['delta_mean'] == 0.


def test_controlled_contrasts_include_untrained_and_pheromone_ablation():
    tables = build([])
    pairs = {(r['method'], r['reference_method']) for r in tables['paired_geometry_costs']}
    assert ('pomo', 'pomo_untrained') in pairs
    assert ('hbg_facs', 'hbg_no_pheromone') in pairs
    assert ('hbg_facs', 'ortools_routing') in pairs


def test_risk_diagnostics_describe_empirical_variation_without_recalibration():
    row = summary('pomo', 11, 20.) | {'cost_variance_population': .04, 'std_travel_minutes': .2}
    record = select(build([row]), 'risk_diagnostics', method='pomo', metric=None)
    assert record['cvar_minus_mean_minutes'] == 2.
    assert record['coefficient_of_variation'] == .01
    assert record['cvar_excess_percent'] == 10.


def test_recourse_comparison_uses_same_plan_and_budget_is_not_duplicated():
    row = summary('aco', 11)
    changed = row | {'policy': 'causal_edd', 'mean_lateness_minutes': 5.}
    tables = stats.build_tables([row, changed], [], [instance()], ['fixed_order', 'causal_edd'], SEEDS, config())
    delta = select(tables, 'recourse_differences', metric='mean_lateness_minutes')
    assert delta['delta_policy_minus_fixed'] == 3.
    assert delta['same_plan_same_scenarios'] and not delta['independent_population_observation']
    assert len(tables['budget_audit']) == 1


def test_confirmatory_readiness_is_blocked_even_with_large_development_registry():
    registry = [instance(str(i), str(i)) for i in range(40)]
    record = build([], registry=registry)['inference_readiness'][0]
    assert record['status'] == 'blocked_development_bank'
    assert not record['bootstrap_or_tests_executed']
    assert not record['confirmatory_pipeline_implemented']


def test_paired_success_includes_failed_solutions_but_blocks_missing_seeds():
    rows = [summary('pomo', 11), summary('pomo', 22, status='no_solution'),
            summary('pomo_untrained', 11), summary('pomo_untrained', 22)]
    audit = [{**instance(), 'method': 'pomo', 'reference_method': 'pomo_untrained', 'status': 'verified'}]
    row = select(build(rows, contrast_audit=audit), 'paired_geometry_success', method='pomo',
                 reference_method='pomo_untrained', metric='scenario_success_per_scheduled_attempt')
    assert row['delta_percentage_points'] == -25.
    assert row['status'] == 'complete_known_cohort'
    rows.pop()
    audit = [{**instance(), 'method': 'pomo', 'reference_method': 'pomo_untrained', 'status': 'verified'}]
    row = select(build(rows, contrast_audit=audit), 'paired_geometry_success', method='pomo',
                 reference_method='pomo_untrained', metric='scenario_success_per_scheduled_attempt')
    assert row['delta_percentage_points'] is None and row['status'] == 'unresolved_attempts'


def test_infeasible_low_cost_does_not_improve_main_cost_comparison():
    row = summary('aco', 11, 1., feasible=False)
    tables = build([row, summary('aco', 22, 20.)])
    main = select(tables, 'case_summary')
    assert main['mean_conditional'] == 20. and main['metric_available'] == 1
    assert main['conditioning'] == 'validated_conservative_feasible_plans'
    assert not main['cohort_complete']
    joint = select(tables, 'joint_performance', metric=None)
    assert joint['mean_travel_conditional_evaluated'] == 10.5
    assert joint['mean_travel_conditional_conservative_feasible'] == 20.


def test_control_comparison_without_verified_design_is_not_attributed():
    rows = [summary(m, seed) for m in ('hbg_facs', 'hbg_no_pheromone') for seed in SEEDS]
    tables = build(rows)
    row = select(tables, 'paired_geometry_costs', method='hbg_facs', reference_method='hbg_no_pheromone')
    assert row['status'] == 'unverified_control_design' and row['delta_mean'] is None
    assert row['common_feasible_geometry_count'] == 0
    row = select(tables, 'paired_geometry_success', method='hbg_facs', reference_method='hbg_no_pheromone',
                 metric='scenario_success_per_scheduled_attempt')
    assert row['delta_percentage_points'] is None


def test_descriptive_exact_reference_averages_all_replicas_without_best_seed_selection():
    rows = [summary('gurobi', 11, 10), summary('gurobi', 22, 30),
            summary('aco', 11, 30), summary('aco', 22, 40)]
    tables = stats.build_tables(rows, [], [instance()], ['fixed_order'], SEEDS,
                               config() | {'reference_method': 'gurobi'})
    row = select(tables, 'pairwise')
    assert row['reference_method'] == 'gurobi'
    assert row['reference_mean'] == 20 and row['method_mean'] == 35 and row['delta_mean'] == 15


def test_interrupted_exact_incumbent_is_evaluated_and_bound_remains_surrogate_only():
    rows = [summary('gurobi', 11, 100, solver_status='interrupted', bound=90.)]
    tables = build(rows)
    row = select(tables, 'gaps', method='gurobi', metric=None, solver_seed=11)
    assert row['candidate_cost_minutes'] == 100 and row['gap_upperbound_pct'] == 10
    assert row['bound_is_equal_single_attempt_runtime_comparison'] is False
    assert row['bound_is_not_stochastic_mean_cost_reference'] is True
    assert select(tables, 'case_summary', method='gurobi')['evaluated'] == 1


def test_plot_keeps_empty_contrast_rows_inside_axis(tmp_path, monkeypatch):
    plt = pytest.importorskip('matplotlib.pyplot')
    from matplotlib.figure import Figure
    observed = []
    original = Figure.savefig
    def inspect_and_save(figure, filename, **kwargs):
        if 'paired_cost' in str(filename):
            for axis in figure.axes:
                lo, hi = axis.get_ylim()
                observed.append((lo, hi, len(axis.get_yticks())))
        return original(figure, filename, **kwargs)
    monkeypatch.setattr(Figure, 'savefig', inspect_and_save)
    stats.save_plots(build([summary('aco', 11), summary('aco', 22)]), tmp_path)
    assert observed and all(lo < 0 and hi >= count - 1 for lo, hi, count in observed)


Writing tests/test_comparison_stats.py


In [12]:
%%writefile tests/test_comparison.py
"""Contratos del lector 06: ejecutar exclusivamente en Google Colab.

Sólo usa inventarios y CSV sintéticos escritos manualmente. No crea bancos,
resuelve rutas ni ejecuta simulaciones. Localmente revisar únicamente AST.
"""
from copy import deepcopy
import csv
import json
import math
from pathlib import Path
import sys

import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'src'))
from svrplab import comparison as c


def config():
    return {
        'schema_version': 'comparison-06-v2', 'role': 'development_only',
        'analysis_mode': 'descriptive_only',
        'expected_methods': ['gurobi', 'aco', 'tabu', 'am_supervised', 'pomo', 'hbg_facs'],
        'reference_method': 'aco',
        'metrics': ['scenario_strict_feasibility', 'mean_travel_minutes',
                    'cvar_travel_minutes', 'mean_lateness_minutes',
                    'mean_depot_overtime_minutes', 'search_total_seconds'],
        'accepted_bank': json.loads((ROOT / 'configs/accepted_bank_00.json').read_text()),
        'method_contract': json.loads((ROOT / 'configs/methods_01_02.json').read_text()),
    }


def inventory_row(run_id, methods, status='compatible', comparison_id='c' * 64):
    return {'run_id': run_id, 'status': status, 'methods': methods,
            'comparison_id': comparison_id}


def scenario_context():
    return {'run_id': 'manual_run', 'comparison_id': 'c' * 64,
            'method': 'aco', 'solver_seed': 11, 'policy': 'fixed_order',
            'instance_sha256': 'a' * 64, 'geometry_group_sha256': 'g' * 64,
            'n_customers': 2}


SEEDS = [2**53 + 1, 2**63 + 17, 2**64 - 1]


def scenario_rows():
    # Costos, retrasos y factibilidad prefijados; no son salidas de un simulador.
    rows = []
    for i, seed in enumerate(SEEDS):
        travel = [10., 20., 40.][i]
        rows.append({**scenario_context(), 'scenario_seed': str(seed),
                     'travel_minutes': travel, 'base_minutes': 8.,
                     'net_execution_delta_minutes': travel - 8.,
                     'waiting_minutes': [0., 2., 4.][i],
                     'lateness_minutes': [0., 5., 0.][i],
                     'depot_overtime_minutes': [0., 0., 3.][i],
                     'customer_violation_fraction': [0., .5, 0.][i],
                     'late_customers': [0, 1, 0][i], 'used_vehicles': 1,
                     'recourse_reorders': i, 'strict_feasible': i == 0,
                     'all_customers_served': True})
    return rows


def scenario_summary():
    return {'scenarios': 3, 'alpha': .5,
            'mean_travel_minutes': 70. / 3., 'cvar_travel_minutes': 100. / 3.,
            'cost_variance_population': 1400. / 9., 'std_travel_minutes': math.sqrt(1400.) / 3.,
            'scenario_strict_feasibility': 1. / 3.,
            'mean_customer_violation_fraction': 1. / 6.,
            'mean_lateness_minutes': 5. / 3., 'mean_depot_overtime_minutes': 1.,
            'mean_waiting_minutes': 2., 'mean_base_minutes': 8.,
            'mean_net_execution_delta_minutes': 46. / 3.,
            'mean_late_customers': 1. / 3., 'mean_recourse_reorders': 1.,
            'used_vehicles': 1, 'evaluation_seconds': 2., 'trace_validation_seconds': .4}


def write_scenarios(path, rows=None, fieldnames=None):
    rows = scenario_rows() if rows is None else rows
    fields = list(rows[0]) if fieldnames is None else fieldnames
    with Path(path).open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return Path(path)


def test_configuration_accepts_descriptive_development_protocol():
    settings = config()
    before = deepcopy(settings)
    c.validate_config(settings)
    assert settings == before


@pytest.mark.parametrize('field,value', [
    ('schema_version', 'methods-v2'), ('role', 'final_test'),
    ('analysis_mode', 'confirmatory'), ('expected_methods', []),
    ('expected_methods', ['aco', 'aco']), ('reference_method', 'unknown'),
    ('metrics', []), ('metrics', ['mean_travel_minutes', 'mean_travel_minutes']),
    ('metrics', ['unsupported_metric']),
])
def test_configuration_rejects_other_protocols_or_invalid_catalogues(field, value):
    with pytest.raises(ValueError):
        c.validate_config(config() | {field: value})


def test_auto_selection_uses_only_unique_compatible_candidates():
    inventory = [inventory_row('run_classic', ['aco', 'tabu']),
                 inventory_row('run_pomo', ['pomo']),
                 inventory_row('run_bad', ['aco'], 'invalid'),
                 inventory_row('run_partial', ['pomo'], 'incomplete'),
                 inventory_row('run_other_bank', ['tabu'], 'incompatible')]
    expected = {'aco': 'run_classic', 'tabu': 'run_classic', 'pomo': 'run_pomo'}
    assert c.select_runs(inventory, config()) == expected
    assert c.select_runs(list(reversed(inventory)), config()) == expected


def test_auto_selection_allows_absent_methods_without_fabricating_runs():
    assert c.select_runs([inventory_row('run_aco', ['aco'])], config()) == {'aco': 'run_aco'}


@pytest.mark.parametrize('inventory', [[], [inventory_row('incomplete', ['aco'], 'incomplete')],
                                     [inventory_row('other', ['unknown'])]])
def test_empty_effective_selection_is_rejected(inventory):
    with pytest.raises(ValueError):
        c.select_runs(inventory, config())


def test_ambiguous_replicates_require_selection_instead_of_latest_or_best():
    first = inventory_row('20260917_old', ['aco']) | {'mean_travel_minutes': 5.}
    latest = inventory_row('20260918_new', ['aco']) | {'mean_travel_minutes': 2.}
    for inventory in [[first, latest], [latest, first]]:
        with pytest.raises(ValueError):
            c.select_runs(inventory, config())
    assert c.select_runs([first, latest], config(), {'aco': '20260917_old'}) == {'aco': '20260917_old'}


def test_explicit_selection_resolves_each_method_without_using_unselected_replica():
    inventory = [inventory_row('classic_a', ['aco', 'tabu']),
                 inventory_row('classic_b', ['aco', 'tabu']),
                 inventory_row('pomo', ['pomo'])]
    selection = {'aco': 'classic_a', 'tabu': 'classic_b', 'pomo': 'pomo'}
    assert c.select_runs(inventory, config(), selection) == selection


@pytest.mark.parametrize('selection', [
    {'aco': 'missing'}, {'aco': 'tabu_only'}, {'unknown': 'classic'},
    {'aco': 'invalid'}, {'aco': 'incomplete'}, {'aco': 'incompatible'},
])
def test_explicit_selection_rejects_unknown_method_run_or_status(selection):
    inventory = [inventory_row('classic', ['aco']), inventory_row('tabu_only', ['tabu']),
                 *[inventory_row(status, ['aco'], status)
                   for status in ['invalid', 'incomplete', 'incompatible']]]
    with pytest.raises(ValueError):
        c.select_runs(inventory, config(), selection)


def test_selection_rejects_mixing_comparison_protocols_even_if_status_is_compatible():
    inventory = [inventory_row('run_aco', ['aco']),
                 inventory_row('run_pomo', ['pomo'], comparison_id='d' * 64)]
    with pytest.raises(ValueError):
        c.select_runs(inventory, config())
    with pytest.raises(ValueError):
        c.select_runs(inventory, config(), {'aco': 'run_aco', 'pomo': 'run_pomo'})


def test_safe_relative_resolves_file_inside_root(tmp_path):
    root = tmp_path / 'run'
    path = root / 'report/summary.json'
    path.parent.mkdir(parents=True)
    path.write_text('[]')
    assert c._safe_relative(root, 'report/summary.json') == path.resolve()


@pytest.mark.parametrize('relative', ['/tmp/outside', '../outside', 'report/../../outside',
                                      'report/../summary.json'])
def test_safe_relative_rejects_absolute_paths_and_parent_traversal(tmp_path, relative):
    with pytest.raises(ValueError):
        c._safe_relative(tmp_path, relative)


def test_safe_relative_rejects_symlink_escape(tmp_path):
    root, outside = tmp_path / 'run', tmp_path / 'outside'
    root.mkdir()
    outside.mkdir()
    (outside / 'summary.json').write_text('[]')
    (root / 'escape').symlink_to(outside, target_is_directory=True)
    with pytest.raises(ValueError):
        c._safe_relative(root, 'escape/summary.json')


def test_csv_keeps_large_scenario_seeds_as_exact_strings_and_typed_metrics(tmp_path):
    path = write_scenarios(tmp_path / 'scenario_metrics.csv')
    rows = c._read_scenarios(path, scenario_context(), SEEDS)
    assert [row['scenario_seed'] for row in rows] == [str(seed) for seed in SEEDS]
    assert all(type(row['scenario_seed']) is str for row in rows)
    assert [row['travel_minutes'] for row in rows] == [10., 20., 40.]
    assert [row['strict_feasible'] for row in rows] == [True, False, False]
    assert all(type(row['used_vehicles']) is int for row in rows)


@pytest.mark.parametrize('field,value', [
    ('travel_minutes', 'nan'), ('base_minutes', 'inf'),
    ('net_execution_delta_minutes', '-inf'), ('lateness_minutes', ''),
    ('strict_feasible', 'true'), ('strict_feasible', '1'),
    ('all_customers_served', 'False'), ('used_vehicles', '1.5'),
    ('late_customers', 'True'), ('recourse_reorders', '-1'),
    ('scenario_seed', '9007199254740993.0'), ('scenario_seed', '9.007199254740993e15'),
])
def test_csv_rejects_nonfinite_malformed_or_noncanonical_values(tmp_path, field, value):
    rows = scenario_rows()
    rows[0][field] = value
    path = write_scenarios(tmp_path / 'scenario_metrics.csv', rows)
    with pytest.raises(ValueError):
        c._read_scenarios(path, scenario_context(), SEEDS)


@pytest.mark.parametrize('damage', ['duplicate', 'missing', 'unexpected'])
def test_csv_requires_exact_scenario_panel_once_each(tmp_path, damage):
    rows = scenario_rows()
    if damage == 'duplicate':
        rows[1]['scenario_seed'] = rows[0]['scenario_seed']
    elif damage == 'missing':
        rows.pop()
    else:
        rows[1]['scenario_seed'] = '7'
    path = write_scenarios(tmp_path / 'scenario_metrics.csv', rows)
    with pytest.raises(ValueError):
        c._read_scenarios(path, scenario_context(), SEEDS)


def test_csv_rejects_context_from_another_run_or_policy(tmp_path):
    rows = scenario_rows()
    rows[1]['policy'] = 'causal_edd'
    path = write_scenarios(tmp_path / 'scenario_metrics.csv', rows)
    with pytest.raises(ValueError):
        c._read_scenarios(path, scenario_context(), SEEDS)


@pytest.mark.parametrize('damage', ['duplicate_header', 'extra_column', 'missing_column'])
def test_csv_rejects_ambiguous_headers_and_row_shape(tmp_path, damage):
    path = write_scenarios(tmp_path / 'scenario_metrics.csv')
    with path.open(newline='', encoding='utf-8') as stream:
        rows = list(csv.reader(stream))
    if damage == 'duplicate_header':
        rows[0].append('travel_minutes')
        for row in rows[1:]:
            row.append('0')
    elif damage == 'extra_column':
        rows[1].append('orphan_value')
    else:
        index = rows[0].index('base_minutes')
        for row in rows:
            row.pop(index)
    with path.open('w', newline='', encoding='utf-8') as stream:
        csv.writer(stream).writerows(rows)
    with pytest.raises(ValueError):
        c._read_scenarios(path, scenario_context(), SEEDS)


def test_aggregates_use_population_variance_and_fractional_cvar_without_simulation():
    rows, summary = scenario_rows(), scenario_summary()
    before = deepcopy(rows), deepcopy(summary)
    recalculated = c._check_aggregates(rows, summary, .5)
    assert recalculated['mean_travel_minutes'] == pytest.approx(70. / 3.)
    assert recalculated['cost_variance_population'] == pytest.approx(1400. / 9.)
    assert recalculated['cvar_travel_minutes'] == pytest.approx(100. / 3.)
    assert recalculated['cvar_travel_minutes'] != 30.  # No promediar ceil(1.5) escenarios.
    assert recalculated['scenario_strict_feasibility'] == pytest.approx(1. / 3.)
    assert (rows, summary) == before


@pytest.mark.parametrize('field,value', [
    ('mean_travel_minutes', 0.), ('cost_variance_population', 700. / 3.),
    ('cvar_travel_minutes', 30.), ('scenario_strict_feasibility', 1.),
    ('scenarios', 4), ('used_vehicles', 2), ('alpha', .95),
    ('mean_travel_minutes', None), ('mean_lateness_minutes', float('nan')),
])
def test_aggregates_reject_tampered_or_null_summary_instead_of_zero_filling(field, value):
    summary = scenario_summary() | {field: value}
    with pytest.raises(ValueError):
        c._check_aggregates(scenario_rows(), summary, .5)


def test_missing_solution_has_no_scenario_aggregate_instead_of_a_zero_cost():
    with pytest.raises(ValueError):
        c._check_aggregates([], {key: None for key in scenario_summary()}, .5)


def offline_model(fold, seed, checkpoint, training, validation):
    return {'fold': {'fold_id': fold}, 'training_seed': seed,
            'model_checkpoint_sha256': checkpoint,
            'training_seconds': training, 'validation_seconds': validation}


def write_offline_index(root, method, training_id, models, teacher_id=None):
    index = root / c.LEARNED[method] / 'manual_bank' / training_id / 'index'
    index.mkdir(parents=True)
    manifest = {'offline_cost_id': training_id, 'models_trained': len(models),
                'training_seconds_unique': sum(model['training_seconds'] for model in models),
                'validation_seconds_included_in_training': sum(model['validation_seconds'] for model in models),
                'teacher_seconds_unique': 0.}
    if teacher_id is not None:
        manifest.pop('teacher_seconds_unique')
        manifest.update(teacher_corpus_id=teacher_id,
                        teacher_costs_unique={'teacher_search_seconds': 5.,
                                             'teacher_preparation_seconds': 1.,
                                             'teacher_validation_seconds': 2.,
                                             'teacher_wall_seconds': 8.})
    (index / 'models.json').write_text(json.dumps(models), encoding='utf-8')
    (index / 'offline_costs.json').write_text(json.dumps(manifest), encoding='utf-8')
    checksum = c.p.seal_bundle(index)
    rows = [{'method': method, 'offline_cost_id': training_id,
             'offline_manifest_sha256': checksum, 'fold_id': model['fold']['fold_id'],
             'training_seed': model['training_seed'],
             'model_checkpoint_sha256': model['model_checkpoint_sha256'],
             'training_seconds': model['training_seconds'],
             'validation_seconds': model['validation_seconds']}
            for model in models]
    return index, rows


def test_offline_costs_deduplicate_models_across_policies_and_keep_validation_included(tmp_path):
    models = [offline_model('geometry_00', 11, 'a' * 64, 8., 2.),
              offline_model('geometry_01', 11, 'b' * 64, 12., 3.)]
    _, rows = write_offline_index(tmp_path, 'pomo', 'manual_training', models)
    repeated = [{**row, 'policy': policy, 'evaluation_status': 'no_solution'}
                for row in rows for policy in ['fixed_order', 'causal_edd']]
    costs, teachers = c._offline_costs(repeated, tmp_path, 'manual_bank')
    assert len(costs) == 1 and teachers == []
    record = costs[0]
    assert record['status'] == 'verified' and record['selected_models'] == 2
    assert record['selected_training_seconds'] == record['manifest_training_seconds'] == 20.
    assert record['selected_validation_seconds_included'] == 5.
    assert record['validation_is_part_of_training'] is True


def test_offline_missing_manifest_is_explicit_without_fabricating_zero_total(tmp_path):
    row = {'method': 'hbg_facs', 'offline_cost_id': 'absent',
           'offline_manifest_sha256': 'a' * 64, 'fold_id': 'geometry_00',
           'training_seed': 11, 'model_checkpoint_sha256': 'b' * 64,
           'training_seconds': 8., 'validation_seconds': 2.}
    costs, teachers = c._offline_costs([row], tmp_path, 'manual_bank')
    assert costs[0]['status'] == 'manifest_missing' and teachers == []
    assert costs[0]['selected_training_seconds'] == 8.
    assert costs[0]['manifest_training_seconds'] is None
    assert costs[0]['manifest_models_trained'] is None


def test_offline_rejects_conflicting_costs_for_repeated_model(tmp_path):
    model = offline_model('geometry_00', 11, 'a' * 64, 8., 2.)
    _, rows = write_offline_index(tmp_path, 'pomo', 'manual_training', [model])
    with pytest.raises(ValueError):
        c._offline_costs([*rows, rows[0] | {'training_seconds': 9.}], tmp_path, 'manual_bank')


def test_offline_costs_reject_corrupt_manifest_instead_of_using_saved_rows(tmp_path):
    model = offline_model('geometry_00', 11, 'a' * 64, 8., 2.)
    index, rows = write_offline_index(tmp_path, 'pomo', 'manual_training', [model])
    (index / 'offline_costs.json').write_text('{}', encoding='utf-8')
    with pytest.raises(ValueError):
        c._offline_costs(rows, tmp_path, 'manual_bank')


def test_teacher_corpus_is_counted_once_when_reused_by_different_training_ids(tmp_path):
    first = offline_model('geometry_00', 11, 'a' * 64, 8., 2.)
    second = offline_model('geometry_01', 11, 'b' * 64, 12., 3.)
    _, a = write_offline_index(tmp_path, 'am_supervised', 'training_a', [first], 'shared_corpus')
    _, b = write_offline_index(tmp_path, 'am_supervised', 'training_b', [second], 'shared_corpus')
    costs, teachers = c._offline_costs(a + b, tmp_path, 'manual_bank')
    assert len(costs) == 2 and len(teachers) == 1
    assert sum(row['selected_training_seconds'] for row in costs) == 20.
    assert teachers[0]['teacher_corpus_id'] == 'shared_corpus'
    assert teachers[0]['teacher_wall_seconds'] == 8.


def test_hbg_ablation_shares_offline_cost_without_double_charge(tmp_path):
    model = offline_model('geometry_00', 11, 'a' * 64, 8., 2.)
    _, rows = write_offline_index(tmp_path, 'hbg_facs', 'shared_hbg', [model])
    rows.append(rows[0] | {'method': 'hbg_no_pheromone'})
    costs, teachers = c._offline_costs(rows, tmp_path, 'manual_bank')
    assert len(costs) == 1 and teachers == []
    assert costs[0]['selected_training_seconds'] == 8.
    assert costs[0]['shared_by_methods'] == 'hbg_facs|hbg_no_pheromone'
    assert costs[0]['charged_once_across_ablations']


def test_untrained_controls_do_not_request_offline_training_manifests(tmp_path):
    costs, teachers = c._offline_costs([{'method': 'pomo_untrained'}, {'method': 'random_masked'}], tmp_path, 'manual_bank')
    assert costs == teachers == []


def test_discussion_table_does_not_present_missing_cost_as_zero():
    tables = c.stats.build_tables([], [], [{'instance_id': 'manual', 'instance_sha256': 'a',
        'geometry_group_sha256': 'g', 'n_customers': 2}], ['fixed_order'], [11, 29], config())
    table = c._discussion_table_text(tables)
    assert '—' in table and '[0.00, 100.00]' in table
    assert 'no un intervalo de confianza' in table
    assert '0/2' in table


def test_predeclared_method_settings_reject_stale_or_extra_parameter():
    expected = {'pomo': {'n_starts': 16, 'completion_search_nodes': 2048}}
    plan = {'methods': ['pomo'], 'config': {'methods': deepcopy(expected)}}
    c._validate_method_settings(plan, expected)
    plan['config']['methods']['pomo']['n_starts'] = 8
    with pytest.raises(ValueError, match='parámetros no preespecificados'):
        c._validate_method_settings(plan, expected)


def controlled_fixture(reference='hbg_no_pheromone'):
    identity = {'instance_id': 'manual', 'instance_sha256': 'a', 'geometry_group_sha256': 'g', 'n_customers': 2}
    method = 'pomo' if reference in ('pomo_untrained', 'random_masked') else 'hbg_facs'
    namespace = 'reinforcement-04' if method == 'pomo' else 'hybrid-05'
    initial = c.p.seed_for(namespace, 'geometry_00', 11)
    settings = {method: {'n_starts': 16}, reference: {'n_starts': 16}}
    if method == 'hbg_facs':
        settings = {method: {'alpha': 1., 'ants': 16}, reference: {'alpha': 0. if reference == 'hbg_no_pheromone' else 1., 'ants': 16}}
    rows, diagnostics = [], []
    for name in (method, reference):
        rows.append({**identity, 'method': name, 'solver_seed': 11, 'training_id': 'same',
            'model_checkpoint_sha256': 'a' * 64, 'fold_id': 'geometry_00', 'training_seed': 11,
            'offline_manifest_sha256': 'b' * 64, 'training_seconds': 0.})
        diagnostics.append({'instance_sha256': 'a', 'method': name, 'solver_seed': 11,
            'solver_metadata_json': json.dumps({'model_config': {'hidden_dim': 128},
                'mask_version': 'same-mask', 'inference_device': 'cpu', 'initialization_seed': initial})})
    sources = {name: {f: 'h' for f in ('attention.py', 'pomo.py', 'reinforcement.py', 'hbg.py', 'hybrid.py', 'baselines.py', '__environment_sha256')}
               for name in (method, reference)}
    return identity, method, settings, rows, diagnostics, sources


def test_pheromone_contrast_requires_same_checkpoint_and_training_identity():
    identity, method, settings, rows, diagnostics, sources = controlled_fixture()
    def audit():
        records = c._controlled_contrast_audit(rows, diagnostics, [identity], [11], settings, sources)
        return next(r for r in records if r['method'] == method and r['reference_method'] == 'hbg_no_pheromone')
    assert audit()['status'] == 'verified'
    rows[1]['model_checkpoint_sha256'] = 'c' * 64
    assert audit()['status'] == 'unverified'
    assert 'trained_checkpoint' in audit()['reason']


@pytest.mark.parametrize('change', ['settings', 'source', 'initialization', 'model', 'mask', 'environment', 'device'])
def test_learning_contrast_requires_matched_inference_source_and_initialization(change):
    identity, method, settings, rows, diagnostics, sources = controlled_fixture('pomo_untrained')
    if change == 'settings':
        settings['pomo_untrained']['n_starts'] = 32
    elif change == 'source':
        sources['pomo_untrained']['attention.py'] = 'different'
    elif change == 'environment':
        sources['pomo_untrained']['__environment_sha256'] = 'different'
    else:
        meta = json.loads(diagnostics[1]['solver_metadata_json'])
        field = {'initialization': 'initialization_seed', 'model': 'model_config', 'mask': 'mask_version', 'device': 'inference_device'}[change]
        meta[field] = 'different'
        diagnostics[1]['solver_metadata_json'] = json.dumps(meta)
    records = c._controlled_contrast_audit(rows, diagnostics, [identity], [11], settings, sources)
    row = next(r for r in records if r['method'] == method and r['reference_method'] == 'pomo_untrained')
    assert row['status'] == 'unverified'


@pytest.mark.parametrize('flag, normalized', [(True, 'time_budget_reached'), (False, 'interrupted'), (None, 'interrupted')])
def test_interrupted_status_requires_recorded_budget_callback_and_preserves_original(flag, normalized):
    solution = {'status': 'interrupted', 'metadata': {'budget_termination_requested': flag}}
    row = c._solution_diagnostics(solution)
    assert row['solver_status'] == 'interrupted'
    assert row['solver_status_normalized'] == normalized
    assert row['iterations'] is None and row['decode_completion_fraction'] is None
    assert json.loads(row['solver_metadata_json']) == solution['metadata']


def test_solution_diagnostics_records_iterations_and_completion_without_imputation():
    solution = {'status': 'feasible', 'metadata': {'iterations': 7, 'completed_iterations': 6,
        'attempts': 20, 'completed_decodes': 3, 'solution_origin': 'edd_paid_fallback'}}
    row = c._solution_diagnostics(solution)
    assert row['iterations'] == 7 and row['completed_iterations'] == 6
    assert row['decode_completion_fraction'] == .15
    assert row['solution_origin'] == 'edd_paid_fallback'


Writing tests/test_comparison.py


In [13]:
# Fixtures de contrato de comparación, ejecutados exclusivamente en Colab.
import uuid
def persist_validation(label, files, status):
    check = WORK / 'method_checks' / BANK_ID / (label + '_' + uuid.uuid4().hex[:12])
    check.mkdir(parents=True)
    for file in files:
        shutil.copy2(file, check / Path(file).name)
    exp.write_json(check / 'status.json', {'status': status, 'bank_id': BANK_ID,
                   'evaluator_sha256': p.file_hash(p.__file__),
                   'code_hashes': {file.name: p.file_hash(file) for file in (RUNTIME / 'svrplab').glob('*.py')}})
    p.seal_bundle(check)
    target = DRIVE_ROOT / 'method_checks' / BANK_ID / check.name
    p.sync_verified(check, target)
    print('Pruebas exportadas:', target)
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(RUNTIME)
TEST_FILES = ['tests/test_comparison_stats.py', 'tests/test_comparison.py']
validation = subprocess.run([sys.executable, '-m', 'pytest', *TEST_FILES, '-q',
                             '--junitxml=validation_tests.xml'], env=test_env,
                            capture_output=True, text=True)
(RUNTIME / 'validation_tests.txt').write_text(validation.stdout + validation.stderr)
print(validation.stdout)
CONTRACT_ARTIFACTS = [RUNTIME / 'validation_tests.txt', RUNTIME / 'validation_tests.xml',
                    *(RUNTIME / name for name in TEST_FILES)]
persist_validation('contract06', CONTRACT_ARTIFACTS, 'passed' if validation.returncode == 0 else 'failed')
if validation.returncode:
    print(validation.stderr)
    raise RuntimeError('Pruebas fallidas: corregir antes de comparar exportaciones.')


........................................................................ [ 61%]
.............................................                            [100%]
- generated xml file: /content/TESIS_COMPARISON_06/runtime/validation_tests.xml -
117 passed in 14.27s

Pruebas exportadas: /content/drive/MyDrive/TESIS-EHBG-FACS/method_checks/pilot00_280553a33c2ab594/contract06_5ce696f24f79


## 5. Inventario y selección de corridas
El inventario describe las exportaciones disponibles y su estado. Una ausencia de
solución, licencia pendiente o evaluación fallida no se convierte en un costo cero
ni se elimina silenciosamente. La comparación conserva cobertura y motivos de ausencia.

La elección automática sólo es admisible cuando un método tiene un único candidato.
Con varias corridas, `select_runs` se detiene y solicita resolver `RUN_SELECTION`.
No combina la mejor semilla de varias corridas ni completa celdas con resultados
seleccionados por desempeño. Mantener el mismo plan de métodos, políticas y semillas.


In [14]:
INVENTORY = cmp.discover_runs(BANK, METHOD_RUNS_ROOT, COMPARISON_CONFIG)
display(pd.DataFrame(INVENTORY))
print('RUN_SELECTION configurado:', RUN_SELECTION)


,run_id,methods,comparison_id,status,reason,report_sha256
0,20260917T023825Z_87dd1d0a,"[aco, tabu]",be14b6eec3db9f4a7ddf301805b3d067d5dab136d97a55...,incompatible,Protocolo distinto: banco/evaluador/runner/obj...,NaN
1,20260917T045359Z_c97d68ef,[am_supervised],4c72030271f5bc11bd45f81eec3ba81d8c1c40ed6a59b7...,incompatible,Protocolo distinto: banco/evaluador/runner/obj...,NaN
2,20260917T051517Z_b84740d5,[pomo],4c72030271f5bc11bd45f81eec3ba81d8c1c40ed6a59b7...,incompatible,Protocolo distinto: banco/evaluador/runner/obj...,NaN
3,20260917T180611Z_92411bba,[],None,incomplete,Falta meta o reporte sellado.,NaN
4,20260917T202127Z_4d297cf7,[hbg_facs],4c72030271f5bc11bd45f81eec3ba81d8c1c40ed6a59b7...,incompatible,Protocolo distinto: banco/evaluador/runner/obj...,NaN
5,20260917T222728Z_d048db31,"[aco, edd, ortools_routing, tabu]",f6445fdea338a62417782bf24a6664275d4752729b1929...,compatible,Meta/reporte verificados; bundles se verifican...,ffd40839636894c9b53b9d45c1a5d41ffb82ced47d49ea...
6,20260918T004829Z_c888d53a,"[pomo, pomo_untrained, random_masked]",f6445fdea338a62417782bf24a6664275d4752729b1929...,compatible,Meta/reporte verificados; bundles se verifican...,72a1df64ebe57885b277bcf9afb2d28177da147456acea...
7,20260918T005406Z_c8c9b0f8,[am_supervised],f6445fdea338a62417782bf24a6664275d4752729b1929...,compatible,Meta/reporte verificados; bundles se verifican...,ca76f83930640775e4b73e8bf4242aae79ce4b84d3aef5...
8,20260918T015823Z_fd78b10e,[hbg_facs],4c72030271f5bc11bd45f81eec3ba81d8c1c40ed6a59b7...,incompatible,Protocolo distinto: banco/evaluador/runner/obj...,NaN
9,20260919T005140Z_481d1c5f,[gurobi],f6445fdea338a62417782bf24a6664275d4752729b1929...,compatible,Meta/reporte verificados; bundles se verifican...,2ef6722f54ac0808e48d0ceaefd79470308e5e4183ecd8...


RUN_SELECTION configurado: {}


In [17]:
RUN_SELECTION["gurobi"] = "20260921T153712Z_ff43d6e0"

SELECTED_RUNS = cmp.select_runs(
    INVENTORY,
    COMPARISON_CONFIG,
    selection=RUN_SELECTION,
)

display(pd.DataFrame([
    {"method": method, "run_id": run_id}
    for method, run_id in sorted(SELECTED_RUNS.items())
]))

ValueError: Varios runs para aco; fije RUN_SELECTION explícitamente: 20260917T222728Z_d048db31, 20260921T154419Z_6c711439

## 6. Compatibilidad, cobertura y reporte descriptivo
Se exigen las mismas identidades de banco/evaluador, contrato, objetivo, presupuesto
y escenarios según el protocolo común. Los hiperparámetros y pesos propios quedan
identificados por procedencia. Exportaciones de protocolos incompatibles no se
fusionan por tener nombres o columnas parecidos.

Los escenarios miden variabilidad condicional dentro de una geometría; las semillas
miden réplicas del método. `case_summary` promedia réplicas disponibles por instancia;
`geometry_summary` promedia esas medias dando el mismo peso a cada instancia de la
geometría. La media de CVaR se calcula sobre los CVaR de cada plan: no mezcla los
costos de distintos planes en un CVaR nuevo. Las políticas se presentan separadas.

Gurobi es la referencia descriptiva de esta revisión final; su elección se declara posterior al diagnóstico del desarrollo. Las corridas históricas conservan ACO. `pairwise` sólo contrasta una instancia/política
cuando ambos métodos tienen todas las réplicas esperadas evaluadas y la métrica
disponible: `delta_mean = método − Gurobi`. La diferencia porcentual usa Gurobi como
denominador y queda vacía si es cero; FR también se expresa en puntos porcentuales.
Las semillas de solver no se emparejan como flujos aleatorios equivalentes.
`scenario_differences` sí empareja escenarios CRN tras promediar réplicas dentro
de cada método, para FR, viaje, tardanza y exceso de retorno. No son observaciones
independientes para inferencia poblacional.

Los costos de `case_summary`, `geometry_summary` y sus contrastes requieren planes
conservadoramente factibles. `joint_performance` conserva también el costo de todas
las evaluaciones con su propio denominador. Interpretar costo junto con factibilidad y cobertura. Una comparación condicionada
a observaciones disponibles debe mostrar su denominador y fallos excluidos; no es
una conclusión sobre todos los intentos. El banco actual no permite separar el
efecto del tamaño del de la geometría, porque hay una sola geometría por tamaño.
Los tiempos offline no se suman repetidamente por política, escenario o fila.
`offline_costs` distingue modelos seleccionados y totales del manifiesto; validación
ya está incluida en entrenamiento. Los profesores se deduplican por corpus en
`teacher_costs`. Un manifiesto ausente queda indicado, sin inventar un total.
`input_environments.json` conserva el hardware/entorno de las corridas: un mismo
presupuesto de segundos no garantiza hardware ni trabajo equivalentes.

`gaps` usa cotas inferiores finitas de Gurobi del mismo problema conservador,
incluidos límites de tiempo con cota disponible. Sólo contrasta candidatos completos
y conservadoramente factibles: `100*(candidato−cota)/candidato`. Queda vacío ante
candidato cero o una cota contradictoria. Es un gap respecto de una cota del modelo
auxiliar. La cota envolvente reúne todas las réplicas del exacto y se identifica como auditoría; no representa una sola corrida de 120 s. No se resta a viaje estocástico. La comparación de métodos promedia todas las réplicas previstas; igualdad numérica no demuestra por sí sola un óptimo ni el costo `c+Q`.


In [ ]:
COMPARISON = cmp.run_comparison(BANK, METHOD_RUNS_ROOT, OUTPUT_ROOT, COMPARISON_CONFIG,
                                selection=SELECTED_RUNS, local_root=LOCAL_ROOT,
                                validation_artifacts=CONTRACT_ARTIFACTS)
REPORT_DIR = Path(COMPARISON['report_dir'])
print('Comparación:', COMPARISON['comparison_run_id'])
print('Reporte:', REPORT_DIR)
print('SHA-256 CHECKSUMS para 08:', p.file_hash(REPORT_DIR / 'CHECKSUMS.json'))
display(COMPARISON['coverage'])
display(COMPARISON['statistical_scope'])


## 7. Tablas, figuras y archivos sellados
El reporte conserva selección, compatibilidad, cobertura, fallos, tablas por geometría,
contrastes descriptivos y procedencia. Los resultados no modifican las corridas de
entrada. `validated` o una verificación de hashes sólo describe integridad/contratos;
no es una afirmación de FR=1 ni de validez confirmatoria.

Las figuras por geometría/política muestran viaje medio y CVaR sobre planes
conservadoramente factibles, FR condicional, cobertura y éxito sobre todos los intentos
programados. Se exportan en PNG a 300 dpi, PDF y SVG.
Las medias de cohortes incompletas se distinguen con un tramado; conservar visible
esa advertencia al exportar una figura.

`joint_performance` cuenta una ausencia de solución observada como éxito cero, pero
no inventa un costo. Los intentos faltantes, licencias pendientes o evaluación
desconocida producen cotas de desconocimiento; **no son intervalos de confianza**.
`paired_geometry_success` contrasta éxito sobre todas las semillas.
`paired_geometry_costs` sólo compara geometrías con todas las instancias y semillas
factibles en ambos métodos: conserva exclusiones y cuenta las geometrías comunes.
`controlled_contrast_audit` exige los mismos parámetros/fuentes, arquitectura, máscara
e inicialización para controles sin entrenar; para HBG sin feromona exige además
exactamente el mismo checkpoint/entrenamiento y sólo permite cambiar alpha de 1 a 0.
Sin esa evidencia se suprime el contraste (`unverified_control_design`) y se conservan
las mediciones individuales. Se rechazan corridas con parámetros de método distintos
a los preespecificados aunque tengan el mismo comparison_id.
Ambas tablas incluyen controles preespecificados: POMO/sin entrenar, POMO/aleatorio,
HBG/sin feromona, HBG/sin entrenar, HBG/OR-Tools y HBG/EDD.

`risk_diagnostics` informa varianza, desviación, coeficiente de variación y CVaR−media
por plan; nunca amplifica ruido para favorecer un método. `recourse_differences`
compara causal_edd y fixed_order sobre el mismo plan/escenarios. `budget_audit`
conserva presupuesto y tiempo real una sola vez por intento, sin duplicar políticas.
`method_provenance` identifica la receta, objetivo, rol y entorno de cada método.
`solution_diagnostics` conserva terminación, contadores de trayectorias neuronales/EDD
y metadatos completos para auditar de dónde provino la solución seleccionada.
`discussion_tables.md` reúne costos, factibilidad, éxito y denominadores en tablas
listas para la discusión, con un guion donde falta medición.
Los pesos compartidos entre HBG y su ablación se cobran una sola vez como costo offline.
La repetición de semillas de EDD/OR-Tools deterministas no crea observaciones
independientes ni variabilidad algorítmica que no exista.

`inference_readiness` bloquea inferencia en este banco de desarrollo. Es una auditoría
de requisitos pendientes; no es un pipeline confirmatorio ni una validación de potencia.

Leer identificadores y semillas como texto al importar CSV. Se muestran hasta 200 filas
de tablas pequeñas y las primeras 100 de tablas grandes, con su número total y ruta.
La vista previa truncada no elimina filas: los CSV exportados permanecen completos.


In [ ]:
table_counts = {name: len(rows) for name, rows in COMPARISON['tables'].items() if isinstance(rows, list)}
table_counts['coverage'] = len(COMPARISON['coverage'])
table_counts['attempts'] = exp.read_json(REPORT_DIR / 'validation.json')['summary_rows']
for path in sorted(REPORT_DIR.glob('*.csv')):
    total = table_counts.get(path.stem)
    limit = 200 if total is not None and total <= 200 else 100
    preview = pd.read_csv(path, dtype=str, keep_default_na=False, nrows=limit)
    print(f'{path.name}: vista previa {len(preview)} de {total if total is not None else "total no registrado"} filas')
    print('CSV completo:', path)
    with pd.option_context('display.max_rows', limit, 'display.max_columns', None):
        display(preview)
display(Markdown((REPORT_DIR / 'discussion_tables.md').read_text()))
for path in sorted(REPORT_DIR.rglob('*.png')):
    print(path.relative_to(REPORT_DIR))
    display(Image(filename=str(path)))
print('Archivos registrados:')
display(COMPARISON['files'])
print('Salida en Drive:', OUTPUT_ROOT / COMPARISON['comparison_run_id'])


```text
MyDrive/TESIS-EHBG-FACS/
  banks/<bank_id>/                         # banco aceptado, sólo lectura
  method_checks/<bank_id>/contract06_*/    # evidencia de pruebas Colab
  method_runs/<bank_id>/<run_id>/          # exportaciones de 01–05, sólo lectura
  comparison_runs/<bank_id>/<analysis_id>/
    report/
      identity.json, input_inventory.json, input_environments.json
      coverage.csv, attempts.csv, case_summary.csv, geometry_summary.csv
      pairwise.csv, scenario_differences.csv, gaps.csv
      offline_costs.csv, teacher_costs.csv, statistical_scope.json
      report.md, validation.json, code/, contract_validation/, figures/
```

Las salidas se sellan y sincronizan con hashes verificados. La selección y la
configuración quedan identificadas para reconstruir el análisis sin reinterpretar
qué corrida se usó. Una comparación nueva no convierte las cuatro geometrías del
piloto en test final. Antes de una comparación confirmatoria se necesita un protocolo
con geometrías independientes reservadas, tamaño muestral justificado y contrastes
prefijados; ampliar escenarios por geometría no sustituye ese requisito.
